# Leische — Context-Aware Sarcasm Detection in Code-Switching Social Media Posts

**Single-notebook model pipeline.** Data comes from the companion **uyam**
repository (collection + LLM-ensemble annotation); this notebook auto-detects
it (§3 below). The methodology reference is [docs/MODEL_PLAN.md](docs/MODEL_PLAN.md);
the training brief is [docs/FABILE_BRIEF.md](docs/FABILE_BRIEF.md).

> **STATUS: TRAIN-READY, labels unvalidated.** The `sarc-v2` export gives
> 15,000 rows / 1,601 positives / 1,393 threads and the §10 train-ready gate
> passes, so §12 runs the real staged ablation. **Every label is
> LLM-ensemble generated** (Fleiss κ 0.376 on sarcasm; human-vs-ensemble κ
> −0.148 on 31 gold items — [docs/ANNOTATION_PROVENANCE.md](docs/ANNOTATION_PROVENANCE.md)),
> so every metric below is *agreement with the ensemble*, not with human
> judgement, and every artifact is stamped `LABEL_AUTHORITY` saying so.

Headless execution (what the training stages were actually run with):

```bash
uv run python tools/run_notebook.py --list
uv run python tools/run_notebook.py --sections 1,2,3,5,6,7,8,8b,12 --stages A
```

**Contents**
1. Setup & environment
2. Configuration (all switches; thesis baseline defaults, upgrades off)
3. Data — auto-detect uyam, load, validate, readiness gate
4. EDA (rerun on every dataset version)
5. Frozen folds (StratifiedGroupKFold, thread-grouped)
6. Context channels — conversational / temporal / retrieval (+ leakage checks)
7. Model — the 5-stage architecture behind ablation flags
8. Training harness · 8b. Plot helpers
9. Smoke test 1 — overfit 16 rows (baseline + full model)
10. Smoke test 2 — tiny-settings 5-fold dry run (baseline)
11. Full context model — gates logged, checkpoint round-trip (smoke)
12. Ablation matrix (8 conditions × 5 folds, staged A/B/C) + significance · 12b. Specification variants (stage D)
13. RQ3 — two-stage sentiment evaluation
14. Verdict & next steps

## 1 · Setup & environment

In [ ]:
import copy
import json
import math
import os
import platform
import random
import subprocess
import sys
import time
import warnings
from dataclasses import asdict, dataclass, field
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from transformers import AutoModel, AutoTokenizer

print(f"python        {sys.version.split()[0]} on {platform.system()} {platform.release()}")
for mod in (torch, transformers, sklearn, pd, np, matplotlib):
    print(f"{mod.__name__:<13} {mod.__version__}")

# Fragmented caching allocator blocks are the usual cause of an OOM that
# nvidia-smi says should not have happened. Set before the first CUDA alloc.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")   # torch >= 2.9 name

assert torch.cuda.is_available(), "CUDA GPU required for the smoke training runs"
props = torch.cuda.get_device_properties(0)
VRAM_GB = props.total_memory / 1024**3
_cap = torch.cuda.get_device_capability(0)
ARCH = f"sm_{_cap[0]}{_cap[1]}"
print(f"GPU: {props.name} | VRAM: {VRAM_GB:.1f} GB | {ARCH} | "
      f"torch {torch.__version__} built for cuda {torch.version.cuda}")

# A wheel without this GPU's arch fails at the FIRST kernel launch with an
# opaque "no kernel image is available", after the data has already loaded.
# Blackwell (sm_120: RTX 50xx) needs torch >= 2.7 on the cu128 index; the
# cu124 wheels this project used to pin stop at sm_90.
_supported = torch.cuda.get_arch_list()
if ARCH not in _supported:
    raise RuntimeError(
        f"torch {torch.__version__} has no kernels for {ARCH} ({props.name}).\n"
        f"  this build supports: {', '.join(_supported)}\n"
        f"  fix: pyproject pins torch on the cu128 index — run `uv sync`, or\n"
        f"       uv pip install torch --index-url https://download.pytorch.org/whl/cu128")

DEVICE = torch.device("cuda")


def cuda_report(tag=""):
    """Peak VRAM since the last reset — the number that says how much room is left."""
    peak = torch.cuda.max_memory_allocated() / 1024**3
    reserved = torch.cuda.max_memory_reserved() / 1024**3
    print(f"  VRAM{' ' + tag if tag else ''}: peak {peak:.2f} GB allocated / "
          f"{reserved:.2f} GB reserved of {VRAM_GB:.1f} GB")
    return peak


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# determinism check: same seed → bit-identical forward pass
def _seeded_forward():
    set_seed(13)
    return nn.Linear(64, 8)(torch.randn(4, 64))

assert torch.equal(_seeded_forward(), _seeded_forward())
print("determinism check PASSED")

ROOT = Path.cwd()
assert (ROOT / "pyproject.toml").exists(), "run this notebook from the repo root"
CACHE = ROOT / "cache"
CACHE.mkdir(exist_ok=True)
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

## 2 · Configuration

One dataclass, every switch from the plan. Defaults are the thesis-committed
baseline (MODEL_PLAN §4); every §9 upgrade is a flag that defaults to OFF so
each manuscript claim stays reproducible with upgrades disabled.

In [ ]:
@dataclass
class Config:
    dataset_version: str = "v1"   # uyam's tag for the full sarc-v2 export
    run_name: str = "dev"

    # stage 1 — shared encoder (§4.1)
    encoder_name: str = "xlm-roberta-base"
    pooling: str = "mean"            # "mean" | "cls" — both verified in §10
    d_model: int = 256
    max_len_target: int = 192
    max_len_context: int = 96
    max_len_selftext: int = 128

    # RQ2 ablation flags (§7.3)
    use_conv: bool = False
    use_temp: bool = False
    use_ret: bool = False

    # stage 2 — conversational + temporal (thesis §3.4(2), §3.4.1 Stage 2)
    conv_role_embeddings: bool = True
    temporal_k: int = 5                      # thesis: "up to five posts by the same author"
    temporal_window_hours: float | None = 48.0  # thesis: "within 48 hours before the target";
                                             # None = unbounded (MODEL_PLAN §4.2). Coverage under
                                             # the 48 h rule is 34.5% of rows vs 45.7% unbounded,
                                             # so both are reported — see §12b.
    temporal_lambda_init: float = 0.0289     # ln2/24 → half-life of one day, in HOURS
    temporal_lambda_learnable: bool = True   # thesis §3.4.1: "λ is a learnable parameter"
    missing_channel: str = "zeros"           # "zeros" | "learned"

    # stage 3 — retrieval (thesis §3.4(3))
    retrieval_encoder: str = "sentence_transformer"  # | "xlmr_cls" (the literal thesis wording)
    retrieval_model: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    retrieval_k: int = 3                     # thesis default, "subject to hyperparameter tuning"
    retrieval_prototypes: bool = False

    # stage 5 — classifier (§4.5)
    mlp_hidden: int = 256
    dropout: float = 0.2

    # training defaults (§4)
    lr_encoder: float = 2e-5
    lr_heads: float = 1e-4
    warmup_ratio: float = 0.1
    max_epochs: int = 10
    patience: int = 3
    batch_size: int = 8
    grad_accum: int = 2
    fp16: bool = True
    grad_clip: float = 1.0
    # Cap the process below total VRAM so a spike raises a catchable OOM
    # instead of starving the display driver and taking the desktop with it.
    # None disables the cap.
    vram_fraction: float | None = 0.85
    dataloader_workers: int = 0      # >0 forks a copy of CONV/TEMP/EMB per worker
    pin_memory: bool = False         # pinned host memory is not swappable
    seeds: list = field(default_factory=lambda: [13, 42, 7])
    class_weighting: str = "inverse_freq"    # per training fold, never global
    label_source: str = "adjudicated"        # | "annotator_majority" — robustness row
                                             # (docs/ANNOTATION_PROVENANCE.md §6.4)
    max_steps_per_epoch: int | None = None   # smoke throttle

    # §9 upgrades — ALL OFF by default
    sample_weighting: bool = False
    sample_weights: dict = field(default_factory=lambda: {
        # unanimous_partial = only 2 of 3 annotator models returned a label
        "unanimous": 1.0, "unanimous_partial": 0.85, "majority": 0.9,
        "adjudicator": 0.7, "human": 1.0})
    soft_labels: bool = False
    focal_loss: bool = False
    focal_gamma: float = 2.0
    weighted_sampler: bool = False
    target_pos_frac: float = 0.25
    aux_cue_heads: bool = False
    aux_polarity_shift: bool = False
    aux_language_head: bool = False
    aux_loss_weight: float = 0.25
    freeze_bottom_layers: int = 0
    layerwise_lr_decay: float | None = None
    temperature_scaling: bool = False
    tune_threshold_on_val: bool = False

    # evaluation (§7)
    n_folds: int = 5
    natural_only_metrics: bool = True

    # smoke: True → every metric/artifact prefixed SMOKE; run_cv refuses
    # smoke=False while the §10 gate fails
    smoke: bool = True
    # keep the fold-0 / first-seed weights under cache/checkpoints/ (§13, demo)
    save_checkpoint: bool = False

    def tag(self) -> str:
        return "SMOKE " if self.smoke else ""

    def to_dict(self) -> dict:
        return asdict(self)


def smoke_cfg(**over) -> Config:
    """Tiny-settings profile: proves the harness, never produces results."""
    base = dict(run_name="smoke", smoke=True, max_len_target=96, max_len_context=64,
                max_len_selftext=64, batch_size=8, grad_accum=1, max_epochs=2,
                patience=1, max_steps_per_epoch=8, seeds=[13])
    base.update(over)
    return Config(**base)


def ablation_cfg(**over) -> Config:
    """Ultra-tiny profile for the 8×5 matrix dry run."""
    base = dict(run_name="ablation", smoke=True, max_len_target=64, max_len_context=48,
                max_len_selftext=48, batch_size=8, grad_accum=1, max_epochs=1,
                patience=1, max_steps_per_epoch=6, seeds=[13])
    base.update(over)
    return Config(**base)


def real_cfg(**over) -> Config:
    """Reportable-run profile: thesis settings, one seed per call, batch sized
    for the 24 GB GPU (FABILE_BRIEF §0b — the batch-8 default was written for
    an 8 GB card). Measured on this GPU: the full model at batch 32 peaks at
    17.3 GB allocated (85% of the 20.3 GB cap) and batch 48 OOMs, so every
    condition runs batch 16 × grad_accum 2 — effective batch 32, identical
    optimiser schedule for all eight conditions, ~9 GB of headroom."""
    base = dict(smoke=False, batch_size=16, grad_accum=2, seeds=[13])
    base.update(over)
    return Config(**base)


def apply_memory_guard(cfg: Config) -> None:
    """Leave the machine usable. An unbounded process on a laptop GPU can push
    the compositor out of VRAM; capping the fraction turns that into a normal
    torch OOM that the run can catch and report."""
    if cfg.vram_fraction is None:
        print("VRAM cap disabled — the process may use the whole device")
        return
    if not torch.cuda.is_available():   # CPU-only session: nothing to cap
        return
    torch.cuda.set_per_process_memory_fraction(cfg.vram_fraction)
    budget = VRAM_GB * cfg.vram_fraction
    print(f"VRAM cap: {cfg.vram_fraction:.0%} of {VRAM_GB:.1f} GB = {budget:.1f} GB "
          f"({VRAM_GB - budget:.1f} GB left for the display and everything else)")


CFG = Config()  # thesis baseline defaults
apply_memory_guard(CFG)
print(f"config ready — thesis defaults, smoke={CFG.smoke}")

## 3 · Data — load, validate, readiness gate

uyam ships the dataset contract directly (`dataset-v1.jsonl`, `corpus-v1.jsonl`,
`dataset_card.json`). Load order: `./data/dataset-vN.jsonl` first, else the
sibling `../uyam/data/annotated/`. The validator still treats `labels.cues`,
`aux.*` and `human_gold` as optional — an older export left them null — but all
of them are populated in `sarc-v2`.

Conversational context must be the **annotator snapshot** (`context.source ==
"annotator_snapshot"`, MODEL_PLAN §11.7); the gate below refuses to train on a
corpus rebuild. The loader prints, every run, which fields are absent, where the
context came from, and `LABEL_AUTHORITY` — who produced the labels.

In [ ]:
LANGUAGES = ("english", "tagalog", "taglish")
SENTIMENTS = ("positive", "neutral", "negative")
RESOLVED_BY = ("unanimous", "unanimous_partial", "majority", "adjudicator", "human")
CUE_KEYS = ("polarity_inversion", "rhetorical_intent", "contextual_incongruity", "hyperbole")

# Fields uyam does not collect. They are null in every row, so the contract
# treats them as optional and the code paths that consume them stay disabled
# rather than fabricating values (docs/UYAM_HANDOFF.md H1, H2, H5, H6).
UNCOLLECTED = ("labels.cues", "aux.tx_sentiment", "aux.lid", "human_gold")


def locate_data(version: str) -> Path:
    local = ROOT / "data"
    if (local / f"dataset-{version}.jsonl").exists():
        return local
    uyam = ROOT.parent / "uyam" / "data" / "annotated"
    if (uyam / f"dataset-{version}.jsonl").exists():
        print(f"using the sibling uyam export: {uyam}")
        return uyam
    raise FileNotFoundError(
        f"dataset-{version}.jsonl not found in {local} or {uyam} — download the "
        "uyam export into ./data/ or clone uyam next to this repo")


def _read_jsonl(path: Path) -> list[dict]:
    return [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]


def validate_rows(rows: list[dict]) -> list[str]:
    """Dataset-contract checks; returns a list of problem strings."""
    problems, seen = [], set()
    for i, r in enumerate(rows):
        w = f"row {i} ({r.get('reddit_fullname', '?')})"
        bad = lambda m: problems.append(f"{w}: {m}")
        for key in ("reddit_fullname", "record_type", "submission_fullname", "created_utc",
                    "author_hash", "text", "labels", "reliability", "aux", "context"):
            if key not in r:
                bad(f"missing {key}")
        rid = r.get("reddit_fullname")
        if rid in seen:
            bad("duplicate reddit_fullname")
        seen.add(rid)
        if r.get("sampling_strategy") not in ("natural", "keyword_oversampled", None):
            bad(f"sampling_strategy={r.get('sampling_strategy')!r}")
        if not str(r.get("text", "")).strip():
            bad("empty text")
        lab = r.get("labels") or {}
        if not isinstance(lab.get("sarcastic"), bool):
            bad("labels.sarcastic not bool")
        if lab.get("language") not in LANGUAGES:
            bad(f"labels.language={lab.get('language')!r}")
        for k in ("literal_sentiment", "intended_sentiment"):
            # null = the sentiment vote never resolved; RQ3 filters these out
            # rather than dropping the row from the sarcasm task (§13)
            if lab.get(k) is not None and lab.get(k) not in SENTIMENTS:
                bad(f"labels.{k}={lab.get(k)!r}")
        if lab.get("cues") is not None:  # optional — uyam does not collect them
            for k in CUE_KEYS:
                if not isinstance(lab["cues"].get(k), bool):
                    bad(f"labels.cues.{k} not bool")
        if (r.get("reliability") or {}).get("resolved_by") not in RESOLVED_BY:
            bad("bad reliability.resolved_by")
        ctx = r.get("context")
        if not isinstance(ctx, dict):
            bad("context missing")
        else:
            if r.get("record_type") == "comment" and ctx.get("submission") is None:
                bad("comment row with null context.submission")
            if r.get("record_type") == "submission" and ctx.get("submission") is not None:
                bad("submission row must have null context.submission")
    return problems


DATA_DIR = locate_data(CFG.dataset_version)
_rows = _read_jsonl(DATA_DIR / f"dataset-{CFG.dataset_version}.jsonl")
_problems = validate_rows(_rows)
assert not _problems, f"{len(_problems)} contract violations; first: {_problems[:3]}"
df = pd.DataFrame(_rows)
# ISO8601: the exports mix second- and microsecond-precision timestamps
df["created_dt"] = pd.to_datetime(df["created_utc"], utc=True, format="ISO8601")

corpus = pd.DataFrame(_read_jsonl(DATA_DIR / f"corpus-{CFG.dataset_version}.jsonl"))
corpus["created_dt"] = pd.to_datetime(corpus["created_utc"], utc=True, format="ISO8601")
card = json.loads((DATA_DIR / "dataset_card.json").read_text(encoding="utf-8"))

sarcastic = df["labels"].map(lambda l: bool(l["sarcastic"]))
language = df["labels"].map(lambda l: l["language"])
strat_key = language + "|sarc=" + sarcastic.astype(str)
# §7.2: null sampling_strategy (pilot) counts as natural; only explicit
# keyword_oversampled rows are excluded from natural-distribution metrics
natural = df["sampling_strategy"].map(lambda s: s != "keyword_oversampled")

# Identity keys on the uyam commit rather than the version string: uyam
# renumbers its own exports, and "v1" has meant two different corpora.
IDENTITY = {k: card.get(k) for k in ("dataset_version", "prompt_version", "uyam_commit")}
IDENTITY["n_rows"] = len(_rows)
IDENTITY_HASH = __import__("hashlib").sha256(
    json.dumps(IDENTITY, sort_keys=True).encode()).hexdigest()[:10]
FOLDS_FILE = RESULTS / f"folds-{CFG.dataset_version}-{IDENTITY_HASH}.json"
print(f"loaded {len(df)} rows, {int(sarcastic.sum())} sarcastic | corpus {len(corpus)} rows")
print(f"dataset identity: {IDENTITY} (hash {IDENTITY_HASH})")

# what the export does NOT carry — printed every run so no downstream cell
# silently assumes a channel exists (docs/UYAM_HANDOFF.md)
HAS_CUES = df["labels"].map(lambda l: l.get("cues") is not None).any()
HAS_TX = df["aux"].map(lambda a: a.get("tx_sentiment") is not None).any()
CTX_SOURCES = set(df["context"].map(lambda c: c.get("source", "annotator_snapshot")))
print(f"uncollected fields  → cues={HAS_CUES}, aux.tx_sentiment={HAS_TX}, "
      f"human_gold={df['human_gold'].notna().any()}")
print(f"conversational context source: {CTX_SOURCES}")
if CTX_SOURCES != {"annotator_snapshot"}:
    print("  WARNING (§11.7): context is rebuilt from the corpus dump, NOT the "
          "snapshot the annotators saw. Report as a limitation until uyam "
          "exports the snapshot (handoff H2).")
_missing_sent = int(df["labels"].map(lambda l: l["intended_sentiment"] is None).sum())
print(f"rows with an unresolved intended_sentiment (excluded from RQ3): {_missing_sent}")

# Every label in this corpus was produced by an LLM ensemble, not a human
# annotator (docs/ANNOTATION_PROVENANCE.md). This block is stamped onto every
# results artifact so no metric can be read as "F1 against human judgement".
_agree = (card.get("agreement") or {})
_gold = _agree.get("gold_vs_ensemble_cohen_kappa") or {}
LABEL_AUTHORITY = {
    "labels_produced_by": "llm_ensemble",
    "annotators": sorted({a["model_key"] for r in _rows
                          for a in (r["reliability"].get("annotators") or [])}),
    "adjudicator": sorted({r["reliability"]["adjudicator"]["model_key"] for r in _rows
                           if r["reliability"].get("adjudicator")}) or None,
    "prompt_version": card.get("prompt_version"),
    "uyam_commit": card.get("uyam_commit"),
    "sarcastic_fleiss_kappa": ((_agree.get("labels") or {}).get("sarcastic") or {}).get("fleiss_kappa"),
    "human_gold": {"n_items": _gold.get("n_gold_items") or 0,
                   "sarcastic_cohen_kappa": _gold.get("sarcastic")},
    "interpretation": ("metrics are measured against LLM-ensemble labels; they quantify "
                       "agreement with the ensemble, not with human judgement, until the "
                       "gold subset validates the labels"),
}
print(f"label authority: {LABEL_AUTHORITY['labels_produced_by']} "
      f"({'+'.join(LABEL_AUTHORITY['annotators'])}"
      f"{' → ' + '+'.join(LABEL_AUTHORITY['adjudicator']) if LABEL_AUTHORITY['adjudicator'] else ''}) "
      f"| sarcasm Fleiss κ={LABEL_AUTHORITY['sarcastic_fleiss_kappa']} "
      f"| gold n={LABEL_AUTHORITY['human_gold']['n_items']}, "
      f"κ={LABEL_AUTHORITY['human_gold']['sarcastic_cohen_kappa']}")

### §10 readiness gate — two tiers

The gate answers two different questions and they have different consequences:

- **train-ready** — is there enough correctly-structured data to train on at
  all? Failing this blocks `run_cv` from any non-smoke run, in code.
- **claim-ready** — do we know what the resulting numbers *mean*? Failing this
  blocks nothing, but stamps `LABEL_AUTHORITY` onto every artifact so no metric
  can be quoted as agreement with human judgement when it is agreement with an
  LLM ensemble. See [docs/ANNOTATION_PROVENANCE.md](docs/ANNOTATION_PROVENANCE.md).

In [ ]:
@dataclass
class GateReport:
    train_checks: list = field(default_factory=list)
    claim_checks: list = field(default_factory=list)

    @property
    def passed(self) -> bool:
        """Train-ready only — claim-ready never blocks a run."""
        return all(ok for _, ok, _ in self.train_checks)

    @property
    def claims_validated(self) -> bool:
        return all(ok for _, ok, _ in self.claim_checks)

    def render(self) -> str:
        lines = ["§10 readiness gate — TRAIN-READY (blocks run_cv):"]
        lines += [f"  [{'PASS' if ok else 'FAIL'}] {n} — {d}" for n, ok, d in self.train_checks]
        lines.append("  => training is legitimate." if self.passed else
                     "  => BLOCKED: SMOKE mode only.")
        lines.append("")
        lines.append("§10 readiness gate — CLAIM-READY (stamps results, never blocks):")
        lines += [f"  [{'PASS' if ok else 'WARN'}] {n} — {d}" for n, ok, d in self.claim_checks]
        lines.append("  => labels are validated against human judgement."
                     if self.claims_validated else
                     "  => metrics quantify agreement with the LLM ensemble, NOT with human"
                     "\n     judgement. Every results file carries LABEL_AUTHORITY saying so.")
        return "\n".join(lines)


def _vnum(v) -> int:
    try:
        return int(str(v).lstrip("sarc-v").lstrip("v") or 0)
    except ValueError:
        return 0


def readiness_gate() -> GateReport:
    g = GateReport()
    pv, n_pos = card.get("prompt_version"), int(sarcastic.sum())

    # ---- tier 1: can we train at all?
    g.train_checks.append(("full corpus under sarc-v2+",
                           _vnum(pv) >= 2 and len(df) >= 1000,
                           f"prompt_version={pv}, {len(df)} rows "
                           f"(uyam_commit={str(card.get('uyam_commit'))[:10]})"))
    g.train_checks.append(("≥400 sarcastic positives", n_pos >= 400,
                           f"{n_pos} positives ({n_pos / len(df):.1%} base rate)"))
    thin = {k: int(v) for k, v in strat_key.value_counts().items() if v < 10}
    g.train_checks.append(("every language×sarcastic cell ≥10", not thin,
                           f"thin cells: {thin}" if thin
                           else f"all cells ≥10 (min {int(strat_key.value_counts().min())})"))
    # §11.7: a corpus rebuild is not what the annotators conditioned on, so a
    # model trained on it is learning from context the labels never saw
    srcs = set(df["context"].map(lambda c: c.get("source", "annotator_snapshot")))
    g.train_checks.append(("conversational context is the annotator snapshot",
                           srcs == {"annotator_snapshot"}, f"context.source={srcs}"))
    g.train_checks.append(("fold file frozen", FOLDS_FILE.exists(), FOLDS_FILE.name))

    # ---- tier 2: do we know what the numbers mean?
    gold = (card.get("agreement") or {}).get("gold_vs_ensemble_cohen_kappa") or {}
    n_gold, k_gold = int(gold.get("n_gold_items") or 0), gold.get("sarcastic")
    g.claim_checks.append(("gold subset ≥250 items", n_gold >= 250, f"{n_gold} labelled"))
    g.claim_checks.append(("human-vs-ensemble sarcasm κ ≥ 0.60",
                           k_gold is not None and k_gold >= 0.60,
                           f"κ={k_gold} on n={n_gold}"
                           + (" — NEGATIVE: the ensemble and the human annotator do not "
                              "agree on the positive class (docs/ANNOTATION_PROVENANCE.md)"
                              if k_gold is not None and k_gold < 0 else "")))
    ens_k = ((card.get("agreement") or {}).get("labels") or {}).get("sarcastic") or {}
    g.claim_checks.append(("inter-annotator sarcasm κ ≥ 0.60",
                           (ens_k.get("fleiss_kappa") or 0) >= 0.60,
                           f"Fleiss κ={ens_k.get('fleiss_kappa')} across "
                           f"{len(ens_k.get('raters') or [])} annotator models"))
    lid = (card.get("agreement") or {}).get("lid_vs_ensemble_language") or {}
    g.claim_checks.append(("automatic LID validated (thesis §3.2.1)",
                           lid.get("agreement") is not None,
                           f"LID-vs-ensemble agreement {lid.get('agreement')} on n={lid.get('n')}"))
    return g


GATE = readiness_gate()
if not GATE.passed:
    print("=" * 72)
    print(" SMOKE MODE — every number below is a harness or architecture check,")
    print(" NOT a reportable result.")
    print("=" * 72)
elif not GATE.claims_validated:
    print("=" * 72)
    print(" TRAINING UNBLOCKED — but labels are UNVALIDATED. Every metric below")
    print(" measures agreement with the LLM ensemble, not with human judgement.")
    print("=" * 72)
print(GATE.render())
CFG.smoke = not GATE.passed   # the gate decides; smoke_cfg()/ablation_cfg() stay smoke regardless
SMOKE = CFG.tag()

## 4 · EDA (rerun on every dataset version — MODEL_PLAN §5)

In [ ]:
cells = pd.crosstab(language, sarcastic)
print(f"{SMOKE}sarcastic: {sarcastic.sum()} / {len(df)} ({sarcastic.mean():.1%})")
print(f"\n{SMOKE}language × sarcastic cells:")
print(cells)
for l in cells.index:
    for s in cells.columns:
        if cells.loc[l, s] < 10:
            print(f"  FLAG: cell ({l}, sarcastic={s}) has {cells.loc[l, s]} rows (<10)")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
cells.plot.bar(ax=axes[0], title=f"{SMOKE}language × sarcastic")
df["sampling_strategy"].fillna("null (→natural)").value_counts().plot.bar(
    ax=axes[1], title=f"{SMOKE}sampling_strategy")
df["reliability"].map(lambda r: r["resolved_by"]).value_counts().plot.bar(
    ax=axes[2], title=f"{SMOKE}resolved_by")
plt.tight_layout(); plt.show()

rel_votes = df["reliability"].map(lambda r: r["sarcasm_votes"])
print(f"{SMOKE}votes × resolution:")
print(pd.crosstab(rel_votes, df["reliability"].map(lambda r: r["resolved_by"])))

### Text lengths vs the stage-1 truncation budgets (192 / 96 / 128 tokens)

In [ ]:
_tok_eda = AutoTokenizer.from_pretrained(CFG.encoder_name)
target_tokens = df["text"].map(lambda t: len(_tok_eda(t, truncation=False)["input_ids"]))
print(f"{SMOKE}target tokens: median {target_tokens.median():.0f}, "
      f"over budget ({CFG.max_len_target}): {(target_tokens > CFG.max_len_target).mean():.1%}")
plt.figure(figsize=(7, 3))
plt.hist(target_tokens, bins=30)
plt.axvline(CFG.max_len_target, color="r", ls="--", label=f"budget {CFG.max_len_target}")
plt.title(f"{SMOKE}target token counts"); plt.legend(); plt.tight_layout(); plt.show()
print("note: text carries collection-time mojibake (e.g. â€™ for ’); labels were "
      "conditioned on the text AS-IS, so it is never 'fixed' (§11.7).")

### Context coverage — conversational and temporal (viability panels)

In [ ]:
conv_stats = pd.DataFrame([{
    "has_submission": r["context"]["submission"] is not None,
    "n_parents": len(r["context"]["parent_chain"] or []),
    "n_replies": len(r["context"]["replies"] or []),
} for _, r in df.iterrows()])
print(f"{SMOKE}% with submission snapshot: {conv_stats['has_submission'].mean():.1%} | "
      f"≥1 parent: {(conv_stats['n_parents'] > 0).mean():.1%} "
      f"(mean chain {conv_stats['n_parents'].mean():.2f}) | "
      f"≥1 reply: {(conv_stats['n_replies'] > 0).mean():.1%}")

_by_author = {a: g.sort_values("created_dt")[["created_dt", "reddit_fullname"]].values.tolist()
              for a, g in corpus.groupby("author_hash", sort=False)}
prior_counts = df.apply(lambda r: sum(
    1 for dt, fid in _by_author.get(r["author_hash"], [])
    if dt < r["created_dt"] and fid != r["reddit_fullname"]), axis=1)
plt.figure(figsize=(7, 3))
plt.hist(prior_counts, bins=range(0, int(prior_counts.max()) + 2))
plt.title(f"{SMOKE}prior posts per target author (temporal-context viability)")
plt.tight_layout(); plt.show()
for k in (1, 3, CFG.temporal_k):
    print(f"{SMOKE}rows with ≥{k} prior posts: {(prior_counts >= k).mean():.1%}")

### Agreement statistics (echoed from dataset_card.json — cite, don't recompute)

In [ ]:
print(pd.DataFrame({k: {"fleiss_kappa": v["fleiss_kappa"],
                         "krippendorff_alpha": v["krippendorff_alpha"]}
                    for k, v in card["agreement"]["labels"].items()}).T)

gold = card["agreement"]["gold_vs_ensemble_cohen_kappa"]
n_gold = gold.get("n_gold_items") or 0
print(f"\nhuman-vs-ensemble Cohen κ on the gold subset (n={n_gold}):")
print(pd.Series({k: v for k, v in gold.items() if k != "n_gold_items"}, name="cohen_kappa"))
if n_gold and (gold.get("sarcastic") or 0) < 0.4:
    print("\n  WARNING: the gold subset does NOT validate the sarcasm labels.")
    print("  `uv run python tools/gold_disagreements.py` prints every disagreeing row")
    print("  with all annotator rationales. docs/ANNOTATION_PROVENANCE.md has the current")
    print("  diagnosis: the adjudicator pass is the source, not the three annotators.")

lid = card["agreement"].get("lid_vs_ensemble_language") or {}
if lid:
    # thesis 3.2.1 promises this number; the export now makes it reportable
    print(f"\nfastText LID vs ensemble language: {lid['agreement']:.4f} on n={lid['n']}")
    hg = card["agreement"].get("lid_vs_human_gold_language") or {}
    if hg:
        print(f"fastText LID vs human gold:        {hg['accuracy']:.4f} on n={hg['n']}")


### Manual read — every sarcastic row with all three annotator rationales

In [ ]:
for _, r in df[sarcastic].head(20).iterrows():   # §5 asks for a 20-row read
    lab = r["labels"]
    print("=" * 100)
    print(f"[{r['reddit_fullname']}] lang={lab['language']} literal={lab['literal_sentiment']} "
          f"intended={lab['intended_sentiment']} votes={r['reliability']['sarcasm_votes']}")
    print(f"TEXT: {r['text'][:300]}")
    for a in r["reliability"]["annotators"]:
        conf = a.get("confidence")
        print(f"  {a['model_key']:<10} sarcastic={a['sarcastic']!s:<5} "
              f"{a['literal_sentiment']}→{a['intended_sentiment']}"
              f"{f' conf={conf:.2f}' if conf is not None else ''} | {a['rationale']}")
    adj = r["reliability"].get("adjudicator")
    if adj:
        # the adjudicator has the final word and overrides the annotator
        # majority on ~35% of the rows it sees — see ANNOTATION_PROVENANCE.md
        print(f"  {adj['model_key']:<10} sarcastic={adj['sarcastic']!s:<5} "
              f"{adj['literal_sentiment']}→{adj['intended_sentiment']} "
              f"(ADJUDICATOR, final) | {adj['rationale']}")

## 5 · Frozen folds (§7.1)

`StratifiedGroupKFold(5)` — stratify `sarcastic×language`, **group by
`submission_fullname`** (thread-mates share context; splitting a thread across
folds is leakage — §11.2). Val is carved from the train side with the same
grouping (≈80/10/10). Frozen once per dataset version; identity-checked on load.

In [ ]:
def make_folds(n_splits: int = 5, seed: int = 13, smoke: bool = True) -> list[dict]:
    y, groups, ids = strat_key.to_numpy(), df["submission_fullname"].to_numpy(), df["reddit_fullname"].to_numpy()
    with warnings.catch_warnings():
        if smoke:
            warnings.filterwarnings("ignore", message="The least populated class")
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        folds = []
        for fold_i, (train_idx, test_idx) in enumerate(sgkf.split(df, y, groups)):
            gss = GroupShuffleSplit(n_splits=1, test_size=1 / 9, random_state=seed + fold_i)
            tr_rel, val_rel = next(gss.split(train_idx, groups=groups[train_idx]))
            folds.append({"fold": fold_i,
                          "train": ids[train_idx[tr_rel]].tolist(),
                          "val": ids[train_idx[val_rel]].tolist(),
                          "test": ids[test_idx].tolist()})
    thread_of = dict(zip(df["reddit_fullname"], df["submission_fullname"]))
    pos_ids = set(df.loc[sarcastic, "reddit_fullname"])
    for fold in folds:
        parts = {p: {thread_of[i] for i in fold[p]} for p in ("train", "val", "test")}
        for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
            assert not parts[a] & parts[b], f"fold {fold['fold']}: thread spans {a}/{b}"
        for p in ("train", "val", "test"):
            if not pos_ids & set(fold[p]):
                msg = f"fold {fold['fold']} {p} has 0 sarcastic positives"
                if smoke:
                    print(f"WARNING (tolerated on pilot): {msg}")
                else:
                    raise ValueError(msg + " — not valid for real training")
    return folds


# The filename carries the identity hash, so two exports can never collide on
# one fold file and nothing has to be deleted by hand when the data changes
# (§11.8). FOLDS_FILE is defined in §3 so the gate can check for it.
if FOLDS_FILE.exists():
    payload = json.loads(FOLDS_FILE.read_text(encoding="utf-8"))
    assert payload["identity"] == IDENTITY, (
        f"{FOLDS_FILE.name} was built for {payload['identity']}, current export is "
        f"{IDENTITY} — identity hash collision, delete the file and re-freeze")
    FOLDS = payload["folds"]
    print(f"loaded frozen folds (identity verified): {FOLDS_FILE}")
else:
    FOLDS = make_folds(CFG.n_folds, seed=13, smoke=CFG.smoke)
    FOLDS_FILE.write_text(json.dumps(
        {"identity": IDENTITY, "n_splits": CFG.n_folds, "seed": 13, "folds": FOLDS},
        indent=2), encoding="utf-8")
    print(f"froze {len(FOLDS)} folds → {FOLDS_FILE}")

# The gate ran in §3, before this cell created the fold file — re-evaluate so a
# first run is not blocked by an artifact it is about to produce.
GATE = readiness_gate()
CFG.smoke = not GATE.passed
SMOKE = CFG.tag()
print(f"\ntrain-ready re-check after freezing folds:")
for name, ok, detail in GATE.train_checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name} — {detail}")
print("  => training is legitimate."
      if GATE.passed else "  => BLOCKED: SMOKE mode only.")

print(f"\n{SMOKE}fold composition (rows / sarcastic positives):")
pos_ids = set(df.loc[sarcastic, "reddit_fullname"])
print(pd.DataFrame([{"fold": f["fold"], **{p: f"{len(f[p])}/{len(pos_ids & set(f[p]))}"
                                           for p in ("train", "val", "test")}} for f in FOLDS]
                   ).to_string(index=False))

author_of = dict(zip(df["reddit_fullname"], df["author_hash"]))
test_authors = [{author_of[i] for i in f["test"]} for f in FOLDS]
multi = sum(1 for a in set().union(*test_authors) if sum(a in s for s in test_authors) > 1)
print(f"\n{SMOKE}author-overlap audit: {multi} authors appear in >1 test fold "
      "(if a few authors dominate later exports, add author_hash to the grouping key)")

## 6 · Context channels (§4.2–§4.3, §6)

- **conversational** — ONLY the embedded snapshot the annotators saw (§11.7)
- **temporal** — corpus posts by the same author strictly BEFORE the target
- **retrieval** — per-fold banks from TRAINING rows only, same-thread excluded
  (§11.1), leakage asserted at build time

In [ ]:
ROLE_SUBMISSION, ROLE_ANCESTOR, ROLE_REPLY = 0, 1, 2
ROLE_NAMES = {0: "submission", 1: "ancestor", 2: "reply"}


@dataclass
class ConvItem:
    text: str
    role: int
    is_submitter: bool


def build_conversational(row) -> list[ConvItem]:
    ctx = row["context"]
    items = []
    sub = ctx.get("submission")
    if sub is not None:
        text = "\n\n".join(t for t in (sub.get("title"), sub.get("selftext")) if t)
        if text.strip():
            items.append(ConvItem(text, ROLE_SUBMISSION, True))
    for p in ctx.get("parent_chain") or []:
        if (p.get("text") or "").strip():
            items.append(ConvItem(p["text"], ROLE_ANCESTOR, bool(p.get("is_submitter"))))
    for rep in ctx.get("replies") or []:
        if (rep.get("text") or "").strip():
            items.append(ConvItem(rep["text"], ROLE_REPLY, bool(rep.get("is_submitter"))))
    return items


@dataclass
class TemporalItem:
    text: str
    delta_hours: float   # thesis §3.4.1 Stage 2: "elapsed time in hours"
    reddit_fullname: str


class TemporalIndex:
    def __init__(self, corpus_df: pd.DataFrame):
        self._by_author = {
            a: list(zip(g["created_dt"], g["reddit_fullname"], g["text"]))
            for a, g in corpus_df[["author_hash", "created_dt", "reddit_fullname", "text"]]
            .sort_values("created_dt").groupby("author_hash", sort=False)}

    def history(self, author_hash, target_dt, target_fullname, k=5,
                window_hours: float | None = 48.0) -> list[TemporalItem]:
        """Up to k posts by the same author strictly BEFORE the target.

        `window_hours` implements the thesis §3.4(2) 48-hour bound; None lifts
        it.  The bound costs real coverage on this corpus (the annotated window
        is 24 days, not the three months §3.1 assumed), so it is a knob and
        both settings are reported — see §12b.
        """
        prior = [(dt, fid, tx) for dt, fid, tx in self._by_author.get(author_hash, [])
                 if dt < target_dt and fid != target_fullname and (tx or "").strip()]
        if window_hours is not None:
            cutoff = target_dt - pd.Timedelta(hours=window_hours)
            prior = [p for p in prior if p[0] >= cutoff]
        return [TemporalItem(tx, (target_dt - dt).total_seconds() / 3600.0, fid)
                for dt, fid, tx in reversed(prior[-k:])]  # most recent first


class RetrievalEmbeddings:
    """Frozen sentence embeddings for all targets, cached under cache/."""

    def __init__(self, fullnames, vectors):
        self.fullnames = list(fullnames)
        self.vectors = vectors / (np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-12)
        self.index = {f: i for i, f in enumerate(self.fullnames)}

    @staticmethod
    def _encode_xlmr_cls(cfg: Config) -> np.ndarray:
        """The thesis §3.4(3) wording, literally: frozen XLM-R [CLS] vectors.

        Kept as an ablation row rather than the default — untuned masked-LM
        [CLS] is a weak cosine-similarity space, and rebuilding the index from
        the *trained* encoder would make the bank non-stationary across epochs.
        """
        tok = AutoTokenizer.from_pretrained(cfg.encoder_name)
        enc = AutoModel.from_pretrained(cfg.encoder_name).to(DEVICE).eval()
        texts, out = df["text"].tolist(), []
        with torch.no_grad():
            for i in range(0, len(texts), 64):
                b = tok(texts[i:i + 64], truncation=True, max_length=cfg.max_len_target,
                        padding=True, return_tensors="pt").to(DEVICE)
                out.append(enc(**b).last_hidden_state[:, 0].float().cpu().numpy())
        del enc
        torch.cuda.empty_cache()
        return np.concatenate(out)

    @classmethod
    def build(cls, cfg: Config) -> "RetrievalEmbeddings":
        key = (cfg.retrieval_model if cfg.retrieval_encoder == "sentence_transformer"
               else cfg.encoder_name + "-cls")
        cache = CACHE / f"retrieval-{cfg.dataset_version}-{key.replace('/', '__')}.npz"
        fullnames = df["reddit_fullname"].tolist()
        if cache.exists():
            z = np.load(cache, allow_pickle=True)
            if z["fullnames"].tolist() == fullnames:
                return cls(fullnames, z["vectors"])
        if cfg.retrieval_encoder == "xlmr_cls":
            vectors = cls._encode_xlmr_cls(cfg)
        else:
            from sentence_transformers import SentenceTransformer
            vectors = SentenceTransformer(cfg.retrieval_model).encode(
                df["text"].tolist(), batch_size=64, show_progress_bar=False, convert_to_numpy=True)
        np.savez_compressed(cache, fullnames=np.array(fullnames, dtype=object), vectors=vectors)
        return cls(fullnames, vectors)


@dataclass
class RetrievalResult:
    sarc: list
    nonsarc: list


# ---- label sources --------------------------------------------------------
# The shipped label is the ADJUDICATED one. docs/ANNOTATION_PROVENANCE.md §3
# traces the negative gold κ to the adjudicator pass, so the annotator
# MAJORITY (what the label would have been without stage 3) is available as a
# robustness row (§4 of the brief): same folds, same everything, other label.
def _majority_sarcastic(r) -> bool:
    votes = [a.get("sarcastic") for a in (r["reliability"].get("annotators") or [])
             if isinstance(a.get("sarcastic"), bool)]
    if not votes or 2 * sum(votes) == len(votes):   # no votes, or a 1–1 tie → shipped label
        return bool(r["labels"]["sarcastic"])
    return 2 * sum(votes) > len(votes)


SARC_MAJORITY = pd.Series([_majority_sarcastic(r) for r in _rows], index=df.index)
LABEL_MAPS = {
    "adjudicated": dict(zip(df["reddit_fullname"], sarcastic.astype(bool))),
    "annotator_majority": dict(zip(df["reddit_fullname"], SARC_MAJORITY.astype(bool))),
}


def label_map(cfg: Config) -> dict:
    return LABEL_MAPS[cfg.label_source]


print(f"label sources — adjudicated (shipped): {int(sarcastic.sum())} positives | "
      f"annotator_majority: {int(SARC_MAJORITY.sum())} positives "
      f"({(SARC_MAJORITY != sarcastic).sum()} rows differ; ANNOTATION_PROVENANCE.md §3)")

# row lookups shared by the audit, the dataset class and the demo export
_rows_by_name = {r["reddit_fullname"]: r for r in _rows}
_text_of = dict(zip(df["reddit_fullname"], df["text"]))
_thread_of = dict(zip(df["reddit_fullname"], df["submission_fullname"]))
_thread_id = {t: i for i, t in enumerate(df["submission_fullname"].unique())}


def assert_no_leakage(retrieval: dict, train_set: set) -> None:
    """§11.1 hard checks: bank ⊆ training fold, never the query's own thread."""
    for q, res in retrieval.items():
        for f in res.sarc + res.nonsarc:
            assert f in train_set, f"retrieval leakage: {f} outside the training fold (query {q})"
            assert _thread_of[f] != _thread_of[q], f"retrieval leakage: {f} shares thread with {q}"
            assert f != q, f"retrieval leakage: {q} retrieved itself"


_RET_CACHE = {}


def build_fold_retrieval(emb: RetrievalEmbeddings, train_fullnames, query_fullnames,
                         k=5, label_source="adjudicated") -> dict:
    """Top-k exemplars from each bank (sarcastic / not) for every query.

    Cosine over unit vectors, banks drawn from the training fold only, the
    query's own thread excluded (which also excludes the query itself).
    Vectorised over queries — the per-query Python loop this replaces cost
    minutes per fold, and the bank is rebuilt for every fold × seed × condition.
    Memoised per (embedding space, label source, bank, queries, k).
    """
    key = (id(emb), label_source, k, hash(tuple(train_fullnames)), hash(tuple(query_fullnames)))
    if key in _RET_CACHE:
        return _RET_CACHE[key]
    label_of = LABEL_MAPS[label_source]
    queries = list(query_fullnames)
    q_idx = np.array([emb.index[q] for q in queries])
    q_thread = np.array([_thread_id[_thread_of[q]] for q in queries])
    out = {q: RetrievalResult([], []) for q in queries}
    for lab in (True, False):
        bank = [f for f in train_fullnames if label_of[f] == lab]
        if not bank:
            continue
        bank_vecs = emb.vectors[[emb.index[f] for f in bank]]
        bank_thread = np.array([_thread_id[_thread_of[f]] for f in bank])
        kk = min(k, len(bank))
        for s in range(0, len(queries), 1024):
            sims = emb.vectors[q_idx[s:s + 1024]] @ bank_vecs.T
            sims[q_thread[s:s + 1024, None] == bank_thread[None, :]] = -np.inf
            top = (np.argpartition(-sims, kk - 1, axis=1)[:, :kk] if kk < sims.shape[1]
                   else np.tile(np.arange(sims.shape[1]), (sims.shape[0], 1)))
            for r in range(sims.shape[0]):
                order = top[r][np.argsort(-sims[r, top[r]], kind="stable")]
                picked = [bank[j] for j in order if np.isfinite(sims[r, j])]
                if lab:
                    out[queries[s + r]].sarc = picked
                else:
                    out[queries[s + r]].nonsarc = picked
    assert_no_leakage(out, set(train_fullnames))
    _RET_CACHE[key] = out
    return out


# Conversational context is fold- and config-independent; the temporal and
# retrieval channels depend on config knobs (§12b varies them), so both are
# memoised per distinct setting rather than rebuilt per fold.
CONV = {r["reddit_fullname"]: build_conversational(r) for _, r in df.iterrows()}
TINDEX = TemporalIndex(corpus)
_TEMP_CACHE, _EMB_CACHE = {}, {}


def build_temporal(cfg: Config) -> dict:
    key = (cfg.temporal_k, cfg.temporal_window_hours)
    if key not in _TEMP_CACHE:
        _TEMP_CACHE[key] = {
            r["reddit_fullname"]: TINDEX.history(
                r["author_hash"], r["created_dt"], r["reddit_fullname"],
                k=cfg.temporal_k, window_hours=cfg.temporal_window_hours)
            for _, r in df.iterrows()}
    return _TEMP_CACHE[key]


def build_embeddings(cfg: Config) -> RetrievalEmbeddings:
    key = (cfg.retrieval_encoder, cfg.retrieval_model, cfg.encoder_name)
    if key not in _EMB_CACHE:
        _EMB_CACHE[key] = RetrievalEmbeddings.build(cfg)
    return _EMB_CACHE[key]


TEMP = build_temporal(CFG)
EMB = build_embeddings(CFG)
_deltas = [it.delta_hours for items in TEMP.values() for it in items]
assert all(d > 0 for d in _deltas), "temporal strictly-before violated"
if CFG.temporal_window_hours is not None:
    assert all(d <= CFG.temporal_window_hours for d in _deltas), "temporal window violated"
_cov = np.mean([len(v) > 0 for v in TEMP.values()])
print(f"{SMOKE}conversational items: {sum(len(v) for v in CONV.values())} | "
      f"temporal items: {len(_deltas)} (>=1 item on {_cov:.1%} of rows, "
      f"k={CFG.temporal_k}, window={CFG.temporal_window_hours}h) | "
      f"retrieval embeddings: {EMB.vectors.shape} ({CFG.retrieval_encoder})")

### Audit — exactly which texts entered each channel (5 random rows, fold-0 banks)

In [ ]:
_fold0 = FOLDS[0]
_ret_demo = build_fold_retrieval(EMB, _fold0["train"],
                                 _fold0["train"] + _fold0["val"] + _fold0["test"],
                                 k=CFG.retrieval_k)
print(f"leakage assertions PASSED for {len(_ret_demo)} queries "
      f"(banks ⊆ {len(_fold0['train'])} train rows, same-thread excluded)")

_clip = lambda t: (" ".join(t.split()))[:110] + ("…" if len(t) > 110 else "")
rng = np.random.default_rng(13)
for f in rng.choice(list(_ret_demo.keys()), size=5, replace=False):
    r = _rows_by_name[f]
    print(f"=== {f} ({r['labels']['language']}, sarcastic={r['labels']['sarcastic']}) ===")
    print(f"TARGET: {_clip(r['text'])}")
    for i, it in enumerate(CONV[f]):
        print(f"  conv {i}. {ROLE_NAMES[it.role]:<10} is_submitter={it.is_submitter} | {_clip(it.text)}")
    for i, it in enumerate(TEMP[f]):
        print(f"  temp {i}. Δt={it.delta_hours:8.2f} h | {_clip(it.text)}")
    for x in _ret_demo[f].sarc:
        print(f"  ret S: {_clip(_text_of[x])}")
    for x in _ret_demo[f].nonsarc:
        print(f"  ret N: {_clip(_text_of[x])}")
    print()

## 7 · Model — the 5-stage architecture (§4), all behind ablation flags

One class: `use_conv=use_temp=use_ret=False` reduces it EXACTLY to the RQ1
context-agnostic baseline (`MLP([t])`), so the comparison can never drift.
Disabled channels are removed from the gate softmax (renormalized), not
zero-filled (§7.3).

In [ ]:
CHANNELS = ("conv", "temp", "ret")


def scatter_items(flat, batch_idx, batch_size):
    """[N, d] flattened items + owner index → padded [B, M, d] + bool mask [B, M].

    Vectorised: a stable sort groups items by owner and a cumulative count
    gives each its slot, so order within an owner is preserved whatever order
    the flat list arrived in. The per-item Python loop this replaces was one
    kernel launch per context item — ~300 per batch at batch 32."""
    d = flat.shape[-1]
    counts = torch.bincount(batch_idx, minlength=batch_size)
    max_items = int(counts.max().item()) if counts.numel() and counts.max() > 0 else 1
    out = flat.new_zeros(batch_size, max_items, d)
    mask = torch.zeros(batch_size, max_items, dtype=torch.bool, device=flat.device)
    if flat.shape[0]:
        order = torch.argsort(batch_idx, stable=True)
        owner = batch_idx[order]
        starts = torch.cumsum(counts, 0) - counts            # first slot of each owner
        slot = torch.arange(owner.numel(), device=flat.device) - starts[owner]
        out[owner, slot] = flat[order]
        mask[owner, slot] = True
    return out, mask


class SharedEncoder(nn.Module):
    """Stage 1: ONE XLM-R for every text unit + pooling + projection (§4.1)."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(cfg.encoder_name)
        self.pooling = cfg.pooling
        self.proj = nn.Linear(self.backbone.config.hidden_size, cfg.d_model)
        if cfg.freeze_bottom_layers > 0:
            for p in self.backbone.embeddings.parameters():
                p.requires_grad = False
            for layer in self.backbone.encoder.layer[:cfg.freeze_bottom_layers]:
                for p in layer.parameters():
                    p.requires_grad = False

    def forward(self, input_ids, attention_mask):
        h = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        if self.pooling == "mean":
            m = attention_mask.unsqueeze(-1).to(h.dtype)
            pooled = (h * m).sum(1) / m.sum(1).clamp(min=1e-6)
        else:
            pooled = h[:, 0]
        return self.proj(pooled)

    def forward_chunked(self, input_ids, attention_mask, chunk=64):
        if input_ids.shape[0] <= chunk:
            return self.forward(input_ids, attention_mask)
        return torch.cat([self.forward(input_ids[i:i + chunk], attention_mask[i:i + chunk])
                          for i in range(0, input_ids.shape[0], chunk)])


class Tokenize:
    def __init__(self, cfg: Config):
        self.tokenizer = AutoTokenizer.from_pretrained(cfg.encoder_name)
        self.budgets = {"target": cfg.max_len_target, "context": cfg.max_len_context,
                        "selftext": cfg.max_len_selftext}

    def __call__(self, texts, kind="context", kinds=None):
        """`kinds` gives a per-text budget (the submission item carries the
        §4.1 selftext budget, parents/replies the context budget); groups are
        tokenised separately and padded to a common width, original order kept."""
        if not texts:
            return {"input_ids": torch.zeros(0, 1, dtype=torch.long),
                    "attention_mask": torch.zeros(0, 1, dtype=torch.long)}
        if kinds is None or len(set(kinds)) == 1:
            k = kind if kinds is None else kinds[0]
            enc = self.tokenizer(texts, truncation=True, max_length=self.budgets[k],
                                 padding=True, return_tensors="pt")
            return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}
        groups = {}
        for i, (t, k) in enumerate(zip(texts, kinds)):
            groups.setdefault(k, []).append((i, t))
        encs = {k: self.tokenizer([t for _, t in items], truncation=True, max_length=self.budgets[k],
                                  padding=True, return_tensors="pt") for k, items in groups.items()}
        width = max(e["input_ids"].shape[1] for e in encs.values())
        pad = self.tokenizer.pad_token_id
        ids = torch.full((len(texts), width), pad, dtype=torch.long)
        att = torch.zeros(len(texts), width, dtype=torch.long)
        for k, items in groups.items():
            e, w = encs[k], encs[k]["input_ids"].shape[1]
            rows = torch.tensor([i for i, _ in items])
            ids[rows, :w] = e["input_ids"]
            att[rows, :w] = e["attention_mask"]
        return {"input_ids": ids, "attention_mask": att}


class TargetAttention(nn.Module):
    """Stage-2/3 block: scaled dot-product attention, target as query.

    `score_bias` is added to the pre-softmax scores. The temporal channel uses
    it to carry −λ·Δt: multiplying the softmax weights by exp(−λ·Δt) and
    renormalising is identical to adding −λ·Δt to the scores, and it is what
    the thesis specifies (§3.4.1 Stage 2 — "attention scores are further
    modulated by an exponential time-decay factor").
    """

    def __init__(self, d):
        super().__init__()
        self.w_q, self.w_k, self.w_v = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.scale = math.sqrt(d)

    def forward(self, target, items, mask, score_bias=None):
        q = self.w_q(target).unsqueeze(1)
        k, v = self.w_k(items), self.w_v(items)
        scores = (q @ k.transpose(1, 2)).squeeze(1) / self.scale
        if score_bias is not None:
            scores = scores + score_bias
        scores = scores.masked_fill(~mask, torch.finfo(scores.dtype).min)
        attn = F.softmax(scores, dim=-1)
        attn = torch.where((~mask.any(-1)).unsqueeze(-1), torch.zeros_like(attn), attn)
        return (attn.unsqueeze(1) @ v).squeeze(1)  # empty rows → zeros


class ContextAwareSarcasmModel(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        d = cfg.d_model
        self.encoder = SharedEncoder(cfg)
        self.active = [c for c, on in zip(CHANNELS, (cfg.use_conv, cfg.use_temp, cfg.use_ret)) if on]

        if cfg.use_conv:
            self.conv_attn = TargetAttention(d)
            if cfg.conv_role_embeddings:
                self.role_emb = nn.Embedding(3, d)
                self.submitter_emb = nn.Embedding(2, d)
        if cfg.use_temp:
            self.temp_attn = TargetAttention(d)
            raw = math.log(math.expm1(cfg.temporal_lambda_init))  # softplus(raw)=λ0
            self.temporal_lambda_raw = nn.Parameter(torch.tensor(raw, dtype=torch.float32),
                                                    requires_grad=cfg.temporal_lambda_learnable)
        if cfg.use_ret:
            self.ret_attn_sarc = TargetAttention(d)
            self.ret_attn_nonsarc = TargetAttention(d)
            self.ret_proj = nn.Linear(2 * d, d)
        if self.active and cfg.missing_channel == "learned":
            self.missing_emb = nn.ParameterDict({c: nn.Parameter(torch.zeros(d)) for c in self.active})
        if len(self.active) >= 2:
            # Stage 4, thesis §3.4.1: "a scalar gate value g_i is computed by
            # passing the concatenation of the target embedding h_t and the
            # context vector c_i through a sigmoid-activated linear layer",
            # then normalised to sum to one.  One gate PER CHANNEL, each
            # conditioned on the target — that target conditioning is the
            # per-instance mechanism the thesis claims, and a joint softmax
            # over the concatenated context vectors cannot express it.
            # Only ACTIVE channels get a gate, so §7.3 ablations renormalise
            # over the surviving sources for free.
            self.gate = nn.ModuleList([nn.Linear(2 * d, 1) for _ in self.active])
        in_dim = d * (2 if self.active else 1)
        self.classifier = nn.Sequential(nn.Linear(in_dim, cfg.mlp_hidden), nn.GELU(),
                                        nn.Dropout(cfg.dropout), nn.Linear(cfg.mlp_hidden, 2))
        if cfg.aux_cue_heads:
            self.cue_head = nn.Linear(in_dim, 4)
        if cfg.aux_polarity_shift:
            self.shift_head = nn.Linear(in_dim, 2)
        if cfg.aux_language_head:
            self.lang_head = nn.Linear(in_dim, 3)
        # RQ3 stage-2 head is trained POST-HOC on frozen features (§13), never here

    @property
    def temporal_lambda(self):
        return F.softplus(self.temporal_lambda_raw)

    def _encode_items(self, part, batch_size):
        if part["input_ids"].shape[0] == 0:
            dev = next(self.parameters()).device
            return (torch.zeros(batch_size, 1, self.cfg.d_model, device=dev),
                    torch.zeros(batch_size, 1, dtype=torch.bool, device=dev))
        flat = self.encoder.forward_chunked(part["input_ids"], part["attention_mask"])
        return scatter_items(flat, part["batch_idx"], batch_size)

    def _apply_missing(self, out, empty, channel):
        """empty: [B] bool — rows whose channel had no items at all."""
        if self.cfg.missing_channel == "learned":
            fill = self.missing_emb[channel].to(out.dtype)
            out = torch.where(empty.unsqueeze(-1), fill.expand_as(out), out)
        return out  # "zeros": attention already yields zeros for empty rows

    def _conv_channel(self, batch, t):
        items, mask = self._encode_items(batch["conv"], t.shape[0])
        if self.cfg.conv_role_embeddings and batch["conv"]["input_ids"].shape[0] > 0:
            extra_flat = self.role_emb(batch["conv"]["role"]) + self.submitter_emb(batch["conv"]["is_submitter"])
            extra, _ = scatter_items(extra_flat, batch["conv"]["batch_idx"], t.shape[0])
            items = items + extra
        return self._apply_missing(self.conv_attn(t, items, mask), ~mask.any(-1), "conv")

    def _temp_channel(self, batch, t):
        items, mask = self._encode_items(batch["temp"], t.shape[0])
        score_bias = None
        if batch["temp"]["input_ids"].shape[0] > 0:
            # thesis §3.4.1 Stage 2 decays the ATTENTION SCORES; MODEL_PLAN §4.2
            # instead scaled the K/V vectors, which is not the same thing —
            # w_k/w_v are affine, so a decayed-to-zero item still contributes
            # w_v's bias and recency weighting stops being monotone in Δt.
            lam_dt, _ = scatter_items(
                (-self.temporal_lambda * batch["temp"]["delta_hours"]).unsqueeze(-1),
                batch["temp"]["batch_idx"], t.shape[0])
            score_bias = lam_dt.squeeze(-1).to(items.dtype)
        return self._apply_missing(self.temp_attn(t, items, mask, score_bias=score_bias),
                                   ~mask.any(-1), "temp")

    def _ret_channel(self, batch, t):
        s_items, s_mask = self._encode_items(batch["ret_sarc"], t.shape[0])
        n_items, n_mask = self._encode_items(batch["ret_nonsarc"], t.shape[0])
        out = self.ret_proj(torch.cat([self.ret_attn_sarc(t, s_items, s_mask),
                                       self.ret_attn_nonsarc(t, n_items, n_mask)], dim=-1))
        # banks pad to different widths (sarcastic bank can hold <k exemplars):
        # combine emptiness per row, never mask | mask
        empty = ~(s_mask.any(-1) | n_mask.any(-1))
        # ret_proj is affine, so an empty bank would emit its bias rather than
        # the zeros the "zeros" missing-channel policy promises (§4.2) — zero
        # it here so "zeros" and "learned" are genuinely different options
        out = torch.where(empty.unsqueeze(-1), torch.zeros_like(out), out)
        return self._apply_missing(out, empty, "ret")

    def forward(self, batch):
        t = self.encoder(batch["target"]["input_ids"], batch["target"]["attention_mask"])
        out = {"target_emb": t}
        chans = []
        for name in self.active:
            c = {"conv": self._conv_channel, "temp": self._temp_channel,
                 "ret": self._ret_channel}[name](batch, t)
            chans.append(c)
            out[f"c_{name}"] = c
        if not self.active:
            feats, out["gates"] = t, None
        elif len(self.active) == 1:
            feats = torch.cat([t, chans[0]], dim=-1)
            out["gates"] = torch.ones(t.shape[0], 1, device=t.device)
        else:
            raw = torch.sigmoid(torch.cat(
                [gate(torch.cat([t, c], dim=-1)) for gate, c in zip(self.gate, chans)], dim=-1))
            g = raw / raw.sum(-1, keepdim=True).clamp(min=1e-6)
            feats = torch.cat([t, sum(g[:, i:i + 1] * chans[i] for i in range(len(chans)))], dim=-1)
            out["gates"] = g
        out["features"] = feats
        out["logits"] = self.classifier(feats)
        if self.cfg.aux_cue_heads:
            out["cue_logits"] = self.cue_head(feats)
        if self.cfg.aux_polarity_shift:
            out["shift_logits"] = self.shift_head(feats)
        if self.cfg.aux_language_head:
            out["lang_logits"] = self.lang_head(feats)
        return out


print("model classes defined")

## 8 · Training harness — loss (§4.5 + §9 flags), fold loop, gate enforcement

In [ ]:
LANG2ID = {"english": 0, "tagalog": 1, "taglish": 2}


class SarcasmDataset(Dataset):
    def __init__(self, fullnames, retrieval, temporal=None, labels=None):
        self.samples = [_rows_by_name[f] for f in fullnames]
        self.retrieval = retrieval or {}
        # temporal depends on temporal_k / temporal_window_hours, so it is
        # passed in rather than read from the module-level TEMP
        self.temporal = TEMP if temporal is None else temporal
        # label source (adjudicated vs annotator majority) is a config knob
        self.labels = LABEL_MAPS["adjudicated"] if labels is None else labels

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        r = self.samples[i]
        f, lab, rel = r["reddit_fullname"], r["labels"], r["reliability"]
        ret = self.retrieval.get(f)
        y = int(self.labels[f])
        p_conf = 0.95 if (rel.get("sarcasm_votes") or "") == "3-0" else 0.75  # §9.1
        return {
            "fullname": f, "target_text": r["text"],
            "conv": CONV.get(f, []), "temp": self.temporal.get(f, []),
            "ret_sarc": [_text_of[x] for x in ret.sarc] if ret else [],
            "ret_nonsarc": [_text_of[x] for x in ret.nonsarc] if ret else [],
            "label": y,
            "soft_pos": p_conf if y else 1.0 - p_conf,
            # uyam collects neither cue labels nor a guaranteed sentiment
            # pair; -1 masks the aux losses for rows that cannot supervise them
            "cues": ([float(lab["cues"][k]) for k in CUE_KEYS]
                     if lab.get("cues") else [-1.0] * len(CUE_KEYS)),
            "shift": (int(lab["literal_sentiment"] != lab["intended_sentiment"])
                      if lab["literal_sentiment"] and lab["intended_sentiment"] else -1),
            "lang": LANG2ID[lab["language"]],
            "resolved_by": rel["resolved_by"],
        }


class Collator:
    def __init__(self, cfg: Config, tok: Tokenize):
        self.cfg, self.tok = cfg, tok

    def _flatten(self, per_sample, extras=None, kinds=None):
        texts, batch_idx, flat_extras, flat_kinds = [], [], [], []
        for b, items in enumerate(per_sample):
            for j, t in enumerate(items):
                texts.append(t)
                batch_idx.append(b)
                if extras is not None:
                    flat_extras.append(extras[b][j])
                if kinds is not None:
                    flat_kinds.append(kinds[b][j])
        out = {**self.tok(texts, kinds=flat_kinds if kinds is not None else None),
               "batch_idx": torch.tensor(batch_idx, dtype=torch.long)}
        if extras is not None:
            out["extras"] = flat_extras
        return out

    def __call__(self, samples):
        cfg = self.cfg
        batch = {
            "fullnames": [s["fullname"] for s in samples],
            "target": self.tok([s["target_text"] for s in samples], kind="target"),
            "labels": torch.tensor([s["label"] for s in samples], dtype=torch.long),
            "soft_pos": torch.tensor([s["soft_pos"] for s in samples]),
            "cues": torch.tensor([s["cues"] for s in samples]),
            "shift": torch.tensor([s["shift"] for s in samples], dtype=torch.long),
            "lang": torch.tensor([s["lang"] for s in samples], dtype=torch.long),
            "sample_weight": torch.tensor(
                [cfg.sample_weights.get(s["resolved_by"], 1.0) if cfg.sample_weighting else 1.0
                 for s in samples]),
        }
        if cfg.use_conv:
            # the submission item (title + selftext) gets the §4.1 selftext
            # budget (128); parents and replies get the context budget (96)
            conv = self._flatten(
                [[it.text for it in s["conv"]] for s in samples],
                [[(it.role, it.is_submitter) for it in s["conv"]] for s in samples],
                [["selftext" if it.role == ROLE_SUBMISSION else "context" for it in s["conv"]]
                 for s in samples])
            ex = conv.pop("extras")
            conv["role"] = torch.tensor([e[0] for e in ex], dtype=torch.long)
            conv["is_submitter"] = torch.tensor([int(e[1]) for e in ex], dtype=torch.long)
            batch["conv"] = conv
        if cfg.use_temp:
            temp = self._flatten([[it.text for it in s["temp"]] for s in samples],
                                 [[it.delta_hours for it in s["temp"]] for s in samples])
            temp["delta_hours"] = torch.tensor(temp.pop("extras"))
            batch["temp"] = temp
        if cfg.use_ret:
            batch["ret_sarc"] = self._flatten([s["ret_sarc"] for s in samples])
            batch["ret_nonsarc"] = self._flatten([s["ret_nonsarc"] for s in samples])
        return batch


def to_device(batch, device):
    return {k: ({kk: (vv.to(device) if torch.is_tensor(vv) else vv) for kk, vv in v.items()}
                if isinstance(v, dict) else (v.to(device) if torch.is_tensor(v) else v))
            for k, v in batch.items()}


def class_weights(train_fullnames, labels=None):
    """Inverse-frequency (neg, pos) on ONE training fold — never global."""
    labels = LABEL_MAPS["adjudicated"] if labels is None else labels
    y = np.array([int(labels[f]) for f in train_fullnames])
    n, n_pos = len(y), int(y.sum())
    if n_pos == 0 or n_pos == n:
        print(f"WARNING: degenerate training fold (n_pos={n_pos}) — equal weights")
        return 1.0, 1.0
    return n / (2.0 * (n - n_pos)), n / (2.0 * n_pos)


def compute_loss(out, batch, cfg, class_w):
    logits, labels = out["logits"], batch["labels"]
    if cfg.soft_labels:
        soft = torch.stack([1.0 - batch["soft_pos"], batch["soft_pos"]], dim=-1)
        per = -(soft * F.log_softmax(logits, dim=-1)).sum(-1) * class_w[labels]
    elif cfg.focal_loss:
        ce = F.cross_entropy(logits, labels, weight=class_w, reduction="none")
        per = ((1 - torch.exp(-ce)) ** cfg.focal_gamma) * ce
    else:
        per = F.cross_entropy(logits, labels, weight=class_w, reduction="none")
    loss = (per * batch["sample_weight"]).mean()
    aux = 0.0
    if cfg.aux_cue_heads:
        cue_mask = batch["cues"] >= 0  # -1 = label not collected
        if cue_mask.any():
            aux = aux + F.binary_cross_entropy_with_logits(
                out["cue_logits"][cue_mask], batch["cues"][cue_mask])
    if cfg.aux_polarity_shift:
        # ignore_index skips rows whose sentiment pair never resolved
        aux = aux + F.cross_entropy(out["shift_logits"], batch["shift"], ignore_index=-1)
    if cfg.aux_language_head:
        aux = aux + F.cross_entropy(out["lang_logits"], batch["lang"])
    return loss + cfg.aux_loss_weight * aux if isinstance(aux, torch.Tensor) else loss


def build_optimizer(model, cfg):
    enc_ids = {id(p) for p in model.encoder.backbone.parameters()}
    enc = [p for p in model.parameters() if id(p) in enc_ids and p.requires_grad]
    heads = [p for p in model.parameters() if id(p) not in enc_ids and p.requires_grad]
    if cfg.layerwise_lr_decay is None:
        groups = [{"params": enc, "lr": cfg.lr_encoder}, {"params": heads, "lr": cfg.lr_heads}]
    else:  # §9.4
        layers = model.encoder.backbone.encoder.layer
        n = len(layers)
        groups = [{"params": heads, "lr": cfg.lr_heads},
                  {"params": [p for p in model.encoder.backbone.embeddings.parameters() if p.requires_grad],
                   "lr": cfg.lr_encoder * cfg.layerwise_lr_decay ** n}]
        groups += [{"params": [p for p in l.parameters() if p.requires_grad],
                    "lr": cfg.lr_encoder * cfg.layerwise_lr_decay ** (n - 1 - i)}
                   for i, l in enumerate(layers)]
    return torch.optim.AdamW(groups)


def build_scheduler(optimizer, total_steps, warmup_ratio):
    warmup = max(1, int(total_steps * warmup_ratio))
    return torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda s: s / warmup if s < warmup
        else max(0.0, (total_steps - s) / max(1, total_steps - warmup)))


def safe_f1(y_true, y_pred):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return float(f1_score(y_true, y_pred, zero_division=0))


def best_threshold(y_true, prob, grid=None):
    """F1-optimal decision threshold. Call it on VALIDATION rows only (§9.8):
    at a 10.7% base rate 0.5 is almost never where F1 peaks, and picking the
    threshold on the test fold would be tuning on the test fold."""
    grid = np.linspace(0.02, 0.98, 97) if grid is None else grid
    y, p = np.asarray(y_true), np.asarray(prob)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        f1s = np.array([f1_score(y, p >= t, zero_division=0) for t in grid])
    return float(grid[int(np.argmax(f1s))]), float(f1s.max())


def fit_temperature(margin, y_true, grid=None):
    """One scalar T minimising validation NLL of sigmoid(margin / T) — §9.8
    temperature scaling. Monotone, so it changes calibration, never ranking."""
    grid = np.exp(np.linspace(np.log(0.25), np.log(8.0), 61)) if grid is None else grid
    m, y = np.asarray(margin, dtype=float), np.asarray(y_true, dtype=float)

    def nll(t):
        z = m / t
        return float(np.mean(np.logaddexp(0.0, -z) * y + np.logaddexp(0.0, z) * (1.0 - y)))
    return float(min(grid, key=nll))


def predict(model, loader, cfg, collect_features=False):
    model.eval()
    rows, feats, targets = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = to_device(batch, DEVICE)
            with torch.autocast("cuda", enabled=cfg.fp16):
                out = model(batch)
            logits = out["logits"].float()
            probs = F.softmax(logits, -1)[:, 1].cpu().numpy()
            margin = (logits[:, 1] - logits[:, 0]).cpu().numpy()   # logit of the positive class
            gates = out["gates"].float().cpu().numpy() if out["gates"] is not None else None
            if collect_features:
                feats.append(out["features"].float().cpu().numpy())
                targets.append(out["target_emb"].float().cpu().numpy())
            for i, f in enumerate(batch["fullnames"]):
                row = {"reddit_fullname": f, "y_true": int(batch["labels"][i]),
                       "prob": float(probs[i]), "margin": float(margin[i]),
                       "pred": int(probs[i] >= 0.5)}
                if gates is not None:
                    for j, name in enumerate(model.active):
                        row[f"gate_{name}"] = float(gates[i, j])
                rows.append(row)
    pred = pd.DataFrame(rows)
    if collect_features:
        pred.attrs["features"] = np.concatenate(feats) if feats else np.zeros((0,))
        # target-only embeddings: the RQ3 stage-1 head is context-free by
        # definition (thesis §3.5 — "applied directly to the original post")
        pred.attrs["target_emb"] = np.concatenate(targets) if targets else np.zeros((0,))
    return pred


def train_model(model, train_loader, val_loader, cfg, class_w=(1.0, 1.0), log_every=0):
    """Early-stopped training on val F1; restores the best weights."""
    model.to(DEVICE)
    optimizer = build_optimizer(model, cfg)
    steps = len(train_loader) if cfg.max_steps_per_epoch is None else min(
        len(train_loader), cfg.max_steps_per_epoch)
    scheduler = build_scheduler(optimizer, max(1, steps * cfg.max_epochs // cfg.grad_accum),
                                cfg.warmup_ratio)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.fp16)
    w = torch.tensor(class_w, dtype=torch.float, device=DEVICE)
    best_f1, best_state, patience_left = -1.0, None, cfg.patience
    hist = {"train_loss": [], "val_f1": [], "epoch_sec": []}
    for epoch in range(cfg.max_epochs):
        t_ep = time.time()
        model.train()
        losses = []
        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(train_loader):
            if cfg.max_steps_per_epoch is not None and step >= cfg.max_steps_per_epoch:
                break
            batch = to_device(batch, DEVICE)
            with torch.autocast("cuda", enabled=cfg.fp16):
                loss = compute_loss(model(batch), batch, cfg, w) / cfg.grad_accum
            scaler.scale(loss).backward()
            if (step + 1) % cfg.grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
            losses.append(float(loss.item()) * cfg.grad_accum)
        val = predict(model, val_loader, cfg)
        val_f1 = safe_f1(val["y_true"].to_numpy(), val["pred"].to_numpy())
        hist["train_loss"].append(float(np.mean(losses)) if losses else float("nan"))
        hist["val_f1"].append(val_f1)
        hist["epoch_sec"].append(round(time.time() - t_ep, 1))
        if log_every:
            print(f"{cfg.tag()}  epoch {epoch}: train_loss={hist['train_loss'][-1]:.4f} "
                  f"val_f1@0.5={val_f1:.4f} ({hist['epoch_sec'][-1]:.0f}s)", flush=True)
        if val_f1 > best_f1:
            best_f1, patience_left = val_f1, cfg.patience
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience_left -= 1
            if patience_left <= 0:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    hist["best_val_f1"] = best_f1
    hist["best_epoch"] = int(np.argmax(hist["val_f1"])) + 1 if hist["val_f1"] else 0
    hist["epochs_run"] = len(hist["val_f1"])
    return hist


def make_loaders(cfg, fold, tok, seed):
    labels = label_map(cfg)
    retrieval = build_fold_retrieval(build_embeddings(cfg), fold["train"],
                                     fold["train"] + fold["val"] + fold["test"],
                                     k=cfg.retrieval_k, label_source=cfg.label_source) if cfg.use_ret else None
    temporal = build_temporal(cfg) if cfg.use_temp else None
    collate = Collator(cfg, tok)
    loaders = {}
    for part in ("train", "val", "test"):
        ds = SarcasmDataset(fold[part], retrieval, temporal, labels)
        if part == "train":
            if cfg.weighted_sampler:  # §9.2
                y = np.array([int(labels[r["reddit_fullname"]]) for r in ds.samples])
                pos_frac = max(y.mean(), 1e-6)
                w = np.where(y == 1, cfg.target_pos_frac / pos_frac,
                             (1 - cfg.target_pos_frac) / max(1 - pos_frac, 1e-6))
                loaders[part] = DataLoader(ds, batch_size=cfg.batch_size, collate_fn=collate,
                                           num_workers=cfg.dataloader_workers,
                                           pin_memory=cfg.pin_memory,
                                           sampler=WeightedRandomSampler(
                                               torch.tensor(w, dtype=torch.double), len(ds), True,
                                               generator=torch.Generator().manual_seed(seed)))
            else:
                loaders[part] = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True,
                                           generator=torch.Generator().manual_seed(seed),
                                           num_workers=cfg.dataloader_workers,
                                           pin_memory=cfg.pin_memory, collate_fn=collate)
        else:
            # evaluation carries no activations for backward — twice the batch is free
            loaders[part] = DataLoader(ds, batch_size=cfg.batch_size * 2, shuffle=False,
                                       num_workers=cfg.dataloader_workers,
                                       pin_memory=cfg.pin_memory, collate_fn=collate)
    return loaders


def safe_metrics(y_true, y_pred):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return {"f1": float(f1_score(y_true, y_pred, zero_division=0)),
                "precision": float(precision_score(y_true, y_pred, zero_division=0)),
                "recall": float(recall_score(y_true, y_pred, zero_division=0)),
                "accuracy": float(accuracy_score(y_true, y_pred)) if len(y_true) else float("nan"),
                "n": int(len(y_true)), "n_pos": int(np.sum(y_true))}


def rank_metrics(y_true, prob):
    """Threshold-free view — AUPRC is the honest single number at a 10.7% base rate."""
    from sklearn.metrics import average_precision_score, roc_auc_score
    y = np.asarray(y_true)
    if len(y) == 0 or y.min() == y.max():
        return {"auprc": float("nan"), "auroc": float("nan")}
    return {"auprc": float(average_precision_score(y, prob)), "auroc": float(roc_auc_score(y, prob))}


META = pd.DataFrame({"reddit_fullname": df["reddit_fullname"], "language": language,
                     "record_type": df["record_type"], "natural": natural,
                     "resolved_by": df["reliability"].map(lambda r: r["resolved_by"]),
                     "sarcasm_votes": df["reliability"].map(lambda r: r.get("sarcasm_votes")),
                     "sarcastic_adjudicated": sarcastic.astype(int),
                     "sarcastic_majority": SARC_MAJORITY.astype(int)})

CHECKPOINTS = CACHE / "checkpoints"   # *.pt is gitignored — never committed


def run_cv(cfg: Config, verbose=True):
    """Folds × seeds loop; writes results/<run_name>/. HARD RULE: refuses
    non-smoke runs while the §10 gate fails.

    Per fold it also (i) picks the F1-optimal threshold on the VALIDATION rows
    and (ii) fits a temperature on the validation margins, then applies both to
    the test rows. `pred` stays the committed 0.5 decision; `pred_tuned` and
    `prob_cal` are the §9.8 variants, reported side by side, never swapped in
    silently. With `save_checkpoint`, the fold-0 / first-seed weights are kept
    under cache/checkpoints/ for §13 and the demo."""
    if not GATE.passed and not cfg.smoke:
        raise RuntimeError("§10 readiness gate FAILED — real training refused:\n" + GATE.render())
    tok = Tokenize(cfg)
    torch.cuda.reset_peak_memory_stats()
    labels = label_map(cfg)
    fold_rows, pred_frames, val_frames, histories = [], [], [], []
    t_run = time.time()
    for fold in FOLDS:
        for seed in cfg.seeds:
            t0 = time.time()
            set_seed(seed)
            loaders = make_loaders(cfg, fold, tok, seed)
            w = class_weights(fold["train"], labels) if cfg.class_weighting == "inverse_freq" else (1.0, 1.0)
            model = ContextAwareSarcasmModel(cfg)
            hist = train_model(model, loaders["train"], loaders["val"], cfg, w,
                               log_every=1 if verbose else 0)
            hist["fold"], hist["seed"] = fold["fold"], seed
            histories.append(hist)          # §8b plots training dynamics from these

            # validation-only decisions: threshold + temperature (§9.8)
            val = predict(model, loaders["val"], cfg)
            thr, val_f1_thr = best_threshold(val["y_true"].to_numpy(), val["prob"].to_numpy())
            temp = fit_temperature(val["margin"].to_numpy(), val["y_true"].to_numpy())
            for frame in (val,):
                frame["fold"], frame["seed"] = fold["fold"], seed
                frame["threshold"], frame["temperature"] = thr, temp
            val_frames.append(val)

            pred = predict(model, loaders["test"], cfg)
            pred["prob_cal"] = 1.0 / (1.0 + np.exp(-pred["margin"].to_numpy() / temp))
            pred["pred_tuned"] = (pred["prob"].to_numpy() >= thr).astype(int)
            pred["threshold"], pred["temperature"] = thr, temp
            pred["fold"], pred["seed"] = fold["fold"], seed
            pred = pred.merge(META, on="reddit_fullname", how="left")
            pred_frames.append(pred)
            y = pred["y_true"].to_numpy()
            m = safe_metrics(y, pred["pred"].to_numpy())
            mt = safe_metrics(y, pred["pred_tuned"].to_numpy())
            fold_rows.append({"fold": fold["fold"], "seed": seed, **m,
                              **{f"{k}_tuned": mt[k] for k in ("f1", "precision", "recall", "accuracy")},
                              "threshold": thr, "temperature": temp,
                              **rank_metrics(y, pred["prob"].to_numpy()),
                              "ece": ece(y, pred["prob"].to_numpy()),
                              "ece_cal": ece(y, pred["prob_cal"].to_numpy()),
                              "best_val_f1": hist["best_val_f1"], "val_f1_tuned": val_f1_thr,
                              "best_epoch": hist["best_epoch"], "epochs_run": hist["epochs_run"],
                              "train_sec": round(time.time() - t0, 1)})
            if verbose:
                print(f"{cfg.tag()}fold {fold['fold']} seed {seed}: test F1@0.5={m['f1']:.4f} "
                      f"F1@val-thr({thr:.2f})={mt['f1']:.4f} AUPRC={fold_rows[-1]['auprc']:.4f} "
                      f"(n_pos={m['n_pos']}) val_f1={hist['best_val_f1']:.4f} "
                      f"best_epoch={hist['best_epoch']}/{hist['epochs_run']} "
                      f"[{(time.time() - t0) / 60:.1f} min]", flush=True)
            if cfg.save_checkpoint and fold["fold"] == 0 and seed == cfg.seeds[0] and not cfg.smoke:
                CHECKPOINTS.mkdir(parents=True, exist_ok=True)
                ck = CHECKPOINTS / f"{cfg.run_name}-fold0-seed{seed}.pt"
                torch.save({"state_dict": model.state_dict(), "config": cfg.to_dict(),
                            "threshold": thr, "temperature": temp, "fold": 0, "seed": seed,
                            "train_fullnames": fold["train"], "dataset_identity": IDENTITY,
                            "label_authority": LABEL_AUTHORITY}, ck)
                if verbose:
                    print(f"  checkpoint → {ck}")
            del model, loaders
            torch.cuda.empty_cache()
    peak = cuda_report(f"peak over {cfg.run_name}") if verbose else torch.cuda.max_memory_allocated() / 1024**3
    fold_metrics = pd.DataFrame(fold_rows)
    predictions = pd.concat(pred_frames, ignore_index=True)
    val_predictions = pd.concat(val_frames, ignore_index=True)
    out_dir = RESULTS / cfg.run_name
    out_dir.mkdir(parents=True, exist_ok=True)
    prefix = "SMOKE-" if cfg.smoke else ""
    predictions.to_csv(out_dir / f"{prefix}predictions.csv", index=False)
    val_predictions.to_csv(out_dir / f"{prefix}val_predictions.csv", index=False)
    try:
        commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True,
                                text=True, check=True).stdout.strip()
    except Exception:
        commit = "unknown"
    (out_dir / f"{prefix}run.json").write_text(json.dumps(
        {"config": cfg.to_dict(), "dataset_identity": IDENTITY, "folds_file": FOLDS_FILE.name,
         "leische_commit": commit, "gate_passed": GATE.passed,
         "claims_validated": GATE.claims_validated,
         "label_authority": LABEL_AUTHORITY,
         "wall_clock_sec": round(time.time() - t_run, 1), "peak_vram_gb": round(float(peak), 2),
         "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0)}, indent=2),
        encoding="utf-8")
    # the caveat travels with the CSV too — these get pasted into slide decks
    fold_metrics.insert(0, "label_authority", LABEL_AUTHORITY["labels_produced_by"])
    fold_metrics.insert(1, "claims_validated", GATE.claims_validated)
    fold_metrics.to_csv(out_dir / f"{prefix}fold_metrics.csv", index=False)
    (out_dir / f"{prefix}histories.json").write_text(json.dumps(histories, indent=2),
                                                     encoding="utf-8")
    return {"fold_metrics": fold_metrics, "predictions": predictions,
            "val_predictions": val_predictions, "histories": histories}


# keys that change WHAT a run measures; everything else (run_name, memory
# guards, post-hoc flags) can differ without invalidating a saved run
_RESULT_KEYS = [k for k in Config.__dataclass_fields__ if k not in (
    "run_name", "vram_fraction", "dataloader_workers", "pin_memory", "save_checkpoint",
    "temperature_scaling", "tune_threshold_on_val", "natural_only_metrics")]


def _signature(cfg_dict):
    return {k: cfg_dict.get(k) for k in _RESULT_KEYS}


def run_exists(cfg: Config) -> bool:
    prefix = "SMOKE-" if cfg.smoke else ""
    rj = RESULTS / cfg.run_name / f"{prefix}run.json"
    if not rj.exists():
        return False
    saved = json.loads(rj.read_text(encoding="utf-8"))
    return (_signature(saved["config"]) == _signature(cfg.to_dict())
            and saved.get("dataset_identity") == IDENTITY)


def load_run(cfg: Config):
    prefix = "SMOKE-" if cfg.smoke else ""
    d = RESULTS / cfg.run_name
    out = {"fold_metrics": pd.read_csv(d / f"{prefix}fold_metrics.csv"),
           "predictions": pd.read_csv(d / f"{prefix}predictions.csv"),
           "histories": json.loads((d / f"{prefix}histories.json").read_text(encoding="utf-8"))}
    vp = d / f"{prefix}val_predictions.csv"
    out["val_predictions"] = pd.read_csv(vp) if vp.exists() else None
    return out


def run_or_load(cfg: Config, verbose=True):
    """Skip-if-done: a finished run with the same result-affecting config and
    the same dataset identity is re-loaded from results/ instead of retrained,
    so the notebook can be re-run top to bottom without repeating GPU hours."""
    if run_exists(cfg):
        print(f"{cfg.tag()}{cfg.run_name}: loaded finished run from results/ (config + identity match)")
        return load_run(cfg)
    return run_cv(cfg, verbose=verbose)


def fold_summary(fm, note=True, tuned=False):
    """mean ± std across folds × seeds (§11.6: never report single-run F1).
    tuned=True reports the val-threshold decision (`*_tuned` columns)."""
    sfx = "_tuned" if tuned else ""
    out = {c: f"{fm[c + sfx].mean():.4f} ± {0.0 if np.isnan(fm[c + sfx].std()) else fm[c + sfx].std():.4f}"
           for c in ("f1", "precision", "recall", "accuracy")}
    if "auprc" in fm:
        out["auprc"] = f"{fm['auprc'].mean():.4f} ± {0.0 if np.isnan(fm['auprc'].std()) else fm['auprc'].std():.4f}"
    if note and not GATE.claims_validated:
        print("NOTE: these measure agreement with the LLM-ensemble labels "
              "(docs/ANNOTATION_PROVENANCE.md), not with human judgement.")
    return pd.DataFrame([{"runs": len(fm), "decision": f"val-tuned thr" if tuned else "0.5", **out}])


def natural_and_all(pred, pred_col="pred"):
    """§7.2 hygiene: natural-only vs all-rows metrics."""
    return pd.DataFrame([
        {"slice": "natural_only", **safe_metrics(pred[pred["natural"]]["y_true"].to_numpy(),
                                                 pred[pred["natural"]][pred_col].to_numpy())},
        {"slice": "all_rows", **safe_metrics(pred["y_true"].to_numpy(), pred[pred_col].to_numpy())}])


def slice_metrics(pred, by, pred_col="pred"):
    return pd.DataFrame([{by: v, **safe_metrics(s["y_true"].to_numpy(), s[pred_col].to_numpy())}
                         for v, s in pred.groupby(by)])


def ece(y_true, prob, n_bins=10):
    if len(prob) == 0:
        return float("nan")
    bins, err = np.linspace(0, 1, n_bins + 1), 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (prob >= lo) & (prob < hi if hi < 1 else prob <= hi)
        if m.sum():
            err += (m.sum() / len(prob)) * abs(prob[m].mean() - y_true[m].mean())
    return float(err)


print("training harness ready")

## 8b · Plots — training dynamics, results, diagnostics

Every panel MODEL_PLAN §7.4 asks for, plus the threshold and calibration views a
10.7% base rate makes mandatory. All of them take the frames `run_cv` already
returns, so any run can be plotted after the fact.

| function | answers |
|---|---|
| `plot_training_curves` | is it learning, and where did early stopping fire? |
| `plot_fold_spread` | is a difference real, or fold-to-fold noise? |
| `plot_threshold_sweep` | **0.5 is the wrong threshold at a 10.7% base rate** — where is F1 maximised? |
| `plot_pr_roc` | precision/recall trade-off against the base-rate floor |
| `plot_confusion` | what kind of errors, counts and row-normalised |
| `plot_calibration` | reliability diagram + ECE (§7.4; RQ3 stage 2 depends on it) |
| `plot_ablation` | RQ2 — the 8 conditions with fold spread |
| `plot_slices` | F1 by language / record_type / resolved_by (§7.4) |
| `plot_gates` | per-instance gate behaviour — thesis-discussion material (§4.4) |
| `plot_bootstrap` | is condition 8 − condition 1 distinguishable from zero? |
| `plot_rq3` | RQ3 — pre- vs post-sarcasm sentiment by slice |

Palette: the first three slots of a CVD-validated categorical set
(blue/orange/aqua, all-pairs ΔE 9.2 deutan, 24.0 normal-vision). Aqua sits below
3:1 on the light surface, so every chart that uses it also ships visible labels
or its numeric table.

In [ ]:
import textwrap

# --- chart theme -----------------------------------------------------------
# Three CVD-validated categorical slots (see the §8b note). Magnitude charts use
# ONE hue with emphasis rather than eight colours — the story is a number, not
# an identity. Never a dual y-axis: two measures of different scale get two axes.
SURFACE, INK, INK_2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e3e3e0"
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"      # blue / orange / aqua
MUTED, GOOD, BAD = "#b9b8b2", "#1baf7a", "#e34948"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.edgecolor": GRID,
    "axes.labelcolor": INK_2, "text.color": INK,
    "xtick.color": INK_2, "ytick.color": INK_2,
    "grid.color": GRID, "grid.linewidth": 1.0, "grid.linestyle": "-",
    "axes.grid": True, "axes.axisbelow": True, "legend.frameon": False,
    "font.size": 9, "axes.titlesize": 10, "figure.dpi": 110,
})


def _ax(ax, title="", xlabel="", ylabel=""):
    """Recessive chrome: hairline solid grid, two spines, no box."""
    ax.set_title(title, color=INK, loc="left", pad=8)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    return ax


def _legend_row(ax, ncol=3, drop=0.20):
    """One legend row under the x-axis. Above the axes it fights the title;
    inside it covers marks. `drop` clears the x tick labels."""
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -drop), ncol=ncol,
              borderaxespad=0, fontsize=8, columnspacing=2.4, handlelength=1.6)


def _headroom(ax, values, pad=1.22):
    """Bar-end labels need somewhere to go that is not the title."""
    top = float(np.nanmax(values)) if len(values) else 1.0
    ax.set_ylim(0, max(top * pad, 1e-6))


# A 2px edge in the SURFACE colour is how matplotlib draws the surface gap
# between touching bars — it is white doing the separating, not an ink border.
BAR = dict(edgecolor=SURFACE, linewidth=2)


def _save(fig, name):
    """Every figure lands in results/figures/ so the manuscript can cite it."""
    out = RESULTS / "figures"
    out.mkdir(parents=True, exist_ok=True)
    prefix = globals().get("FIG_PREFIX", "SMOKE-" if CFG.smoke else "")
    path = out / f"{prefix}{name}.png"
    fig.savefig(path, bbox_inches="tight")
    return path


# --- training dynamics -----------------------------------------------------
def plot_training_curves(histories, title=""):
    """Train loss and val F1 per epoch. Two panels, never one dual axis."""
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4))
    for key, ax, colour, label in (("train_loss", axes[0], C1, "train loss"),
                                   ("val_f1", axes[1], C2, "validation F1")):
        for h in histories:
            ax.plot(range(1, len(h[key]) + 1), h[key], color=colour,
                    lw=2, alpha=0.35, solid_capstyle="round")
        longest = max(len(h[key]) for h in histories)
        mean = [float(np.mean([h[key][e] for h in histories if len(h[key]) > e]))
                for e in range(longest)]
        ax.plot(range(1, longest + 1), mean, color=colour, lw=2.5,
                marker="o", ms=5, mec=SURFACE, mew=2, label=f"mean {label}")
        _ax(ax, f"{SMOKE}{label}", "epoch", "")
        _legend_row(ax, ncol=1, drop=0.24)
    # where early stopping actually fired, per run
    stops = [int(np.argmax(h["val_f1"])) + 1 for h in histories]
    axes[1].annotate(f"best epoch: {', '.join(map(str, stops))}",
                     xy=(0.98, 0.04), xycoords="axes fraction", ha="right",
                     color=INK_2, fontsize=8)
    fig.suptitle(title, color=INK, x=0.005, ha="left", fontsize=11) if title else None
    plt.tight_layout(); path = _save(fig, f"training-curves{'-' + title.split()[0] if title else ''}"); plt.show()
    return path


def plot_fold_spread(fold_metrics, metric="f1", title=""):
    """Per-fold/seed points against the mean — is a gap real or noise?"""
    fig, ax = plt.subplots(figsize=(7, 3.2))
    vals = fold_metrics[metric].to_numpy()
    x = fold_metrics["fold"].to_numpy() if "fold" in fold_metrics else np.arange(len(vals))
    ax.scatter(x, vals, s=64, color=C1, zorder=3, edgecolor=SURFACE, linewidth=2)
    mean, std = float(np.mean(vals)), float(np.std(vals))
    ax.axhline(mean, color=INK_2, lw=2, zorder=2)
    ax.axhspan(mean - std, mean + std, color=C1, alpha=0.10, zorder=1)
    ax.annotate(f"mean {mean:.3f} ± {std:.3f}", xy=(0.99, 0.95), xycoords="axes fraction",
                ha="right", va="top", color=INK, fontsize=9)
    _ax(ax, title or f"{SMOKE}{metric} across folds × seeds", "fold", metric)
    ax.set_xticks(sorted(set(x.tolist())))
    plt.tight_layout(); path = _save(fig, f"fold-spread-{metric}"); plt.show()
    return path


# --- threshold, PR/ROC, confusion, calibration -----------------------------
def plot_threshold_sweep(pred, title=""):
    """P / R / F1 against the decision threshold.

    At a ~10% base rate the 0.5 default is almost never F1-optimal (§9.8); this
    is the chart that says where to put it. Returns the optimal threshold.
    """
    y, p = pred["y_true"].to_numpy(), pred["prob"].to_numpy()
    ts = np.linspace(0.02, 0.98, 97)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        curves = {
            "precision": [precision_score(y, p >= t, zero_division=0) for t in ts],
            "recall": [recall_score(y, p >= t, zero_division=0) for t in ts],
            "F1": [f1_score(y, p >= t, zero_division=0) for t in ts]}
    best = float(ts[int(np.argmax(curves["F1"]))])
    fig, ax = plt.subplots(figsize=(7.5, 3.4))
    for (name, ys), colour in zip(curves.items(), (C1, C2, C3)):
        ax.plot(ts, ys, color=colour, lw=2, label=name, solid_capstyle="round")
    ax.axvline(0.5, color=MUTED, lw=1.5, ls="--")
    ax.axvline(best, color=INK, lw=1.5, ls="--")
    ax.annotate(f"F1-optimal {best:.2f}\n(default 0.50)", xy=(best, 0.02),
                xytext=(6, 0), textcoords="offset points", color=INK, fontsize=8)
    _ax(ax, title or f"{SMOKE}metric vs decision threshold", "threshold", "")
    ax.set_ylim(0, 1.02); _legend_row(ax)
    plt.tight_layout(); path = _save(fig, f"threshold-sweep{'-' + title.split()[0] if title else ''}"); plt.show()
    print(f"{SMOKE}F1-optimal threshold {best:.3f} "
          f"(F1 {max(curves['F1']):.4f}) vs 0.50 (F1 {f1_score(y, p >= 0.5, zero_division=0):.4f})")
    return best


def plot_pr_roc(pred, title=""):
    from sklearn.metrics import (average_precision_score, precision_recall_curve,
                                 roc_auc_score, roc_curve)
    y, p = pred["y_true"].to_numpy(), pred["prob"].to_numpy()
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
    prec, rec, _ = precision_recall_curve(y, p)
    axes[0].plot(rec, prec, color=C1, lw=2)
    base = float(np.mean(y))
    axes[0].axhline(base, color=MUTED, lw=1.5, ls="--")
    axes[0].annotate(f"base rate {base:.3f}", xy=(0.02, base), xytext=(0, 5),
                     textcoords="offset points", color=INK_2, fontsize=8)
    _ax(axes[0], f"{SMOKE}precision–recall  ·  AP {average_precision_score(y, p):.3f}",
        "recall", "precision")
    fpr, tpr, _ = roc_curve(y, p)
    axes[1].plot(fpr, tpr, color=C2, lw=2)
    axes[1].plot([0, 1], [0, 1], color=MUTED, lw=1.5, ls="--")
    _ax(axes[1], f"{SMOKE}ROC  ·  AUC {roc_auc_score(y, p):.3f}",
        "false positive rate", "true positive rate")
    for ax in axes:
        ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
    plt.tight_layout(); path = _save(fig, f"pr-roc{'-' + title if title else ''}"); plt.show()
    return path


def plot_confusion(pred, title="", pred_col="pred"):
    y, yh = pred["y_true"].to_numpy(), pred[pred_col].to_numpy()
    cm = confusion_matrix(y, yh, labels=[0, 1])
    norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.2))
    names = ["not sarcastic", "sarcastic"]
    for ax, mat, fmt, sub in ((axes[0], cm, "{:,}", "counts"),
                              (axes[1], norm, "{:.1%}", "row-normalised")):
        ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)   # one hue = magnitude
        for i in range(2):
            for j in range(2):
                ax.text(j, i, fmt.format(mat[i, j]), ha="center", va="center",
                        color=INK if norm[i, j] < 0.55 else SURFACE, fontsize=10)
        ax.set_xticks([0, 1], names, fontsize=8)
        ax.set_yticks([0, 1], names, fontsize=8, rotation=90, va="center")
        _ax(ax, f"{SMOKE}{title + ' — ' if title else ''}{sub}", "predicted", "true")
        ax.grid(False)
    plt.tight_layout(); path = _save(fig, f"confusion{'-' + title if title else ''}"); plt.show()
    return path


def plot_calibration(pred, n_bins=10, title="", prob_col="prob"):
    """Reliability diagram + ECE (§7.4). RQ3 stage 2 only fires on flagged rows,
    so a badly calibrated flag breaks the sentiment story before it starts."""
    y, p = pred["y_true"].to_numpy(), pred[prob_col].to_numpy()
    edges = np.linspace(0, 1, n_bins + 1)
    xs, ys, ns = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (p >= lo) & (p < hi if hi < 1 else p <= hi)
        if m.sum():
            xs.append(p[m].mean()); ys.append(y[m].mean()); ns.append(int(m.sum()))
    # Reliability curve above, bin counts below on a shared x — a number
    # printed beside every point is unreadable, and bin weight still has to
    # be visible or a bin of 10 looks as solid as a bin of 900.
    fig, axes = plt.subplots(2, 1, figsize=(5.4, 4.8), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1], "hspace": 0.12})
    axes[0].plot([0, 1], [0, 1], color=MUTED, lw=1.5, ls="--")
    axes[0].plot(xs, ys, color=C1, lw=2, marker="o", ms=8, mec=SURFACE, mew=2)
    _ax(axes[0], title or f"{SMOKE}reliability  ·  ECE {ece(y, p):.4f}",
        "", "observed positive rate")
    axes[0].set_ylim(0, 1.04)
    axes[1].bar(xs, ns, width=(1.0 / n_bins) * 0.82, color=MUTED, **BAR)
    _ax(axes[1], "", "mean predicted probability", "rows")
    axes[1].set_xlim(0, 1)
    plt.tight_layout(); path = _save(fig, f"calibration{'-' + prob_col if prob_col != 'prob' else ''}"
                                     f"{'-' + title.split()[0] if title else ''}"); plt.show()
    return path


# --- RQ2 / slices / gates --------------------------------------------------
def plot_ablation(ablation, metric="f1", title=""):
    """RQ2's 8 conditions. ONE hue with emphasis — the story is a magnitude, not
    eight identities; baseline and full model are the two bars that matter."""
    names = list(ablation.keys())
    means = [ablation[n]["fold_metrics"][metric].mean() for n in names]
    stds = [np.nan_to_num(ablation[n]["fold_metrics"][metric].std()) for n in names]
    colours = [C2 if n.startswith(("1_", "8_")) else MUTED for n in names]
    fig, ax = plt.subplots(figsize=(9, 3.6))
    ax.bar(range(len(names)), means, yerr=stds, width=0.62, color=colours,
           capsize=4, error_kw={"ecolor": INK_2, "elinewidth": 1.2}, **BAR)
    for i, (m, s) in enumerate(zip(means, stds)):
        ax.annotate(f"{m:.3f}", xy=(i, m + s), xytext=(0, 4), textcoords="offset points",
                    ha="center", color=INK, fontsize=8)
    ax.axhline(means[0], color=C2, lw=1.5, ls="--", alpha=0.6)
    _ax(ax, title or f"{SMOKE}RQ2 ablation — {metric} (mean ± std over folds × seeds)",
        "", metric)
    _headroom(ax, np.asarray(means) + np.asarray(stds), pad=1.16)
    ax.set_xticks(range(len(names)), [n.split("_", 1)[-1] for n in names],
                  rotation=20, ha="right", fontsize=8)
    plt.tight_layout(); path = _save(fig, f"ablation-{metric}"); plt.show()
    return path


def plot_slices(pred, by, metric="f1", title="", pred_col="pred"):
    """§7.4 disaggregation. n is printed on every bar — a slice of 12 rows and a
    slice of 3,000 must not look equally authoritative."""
    tab = slice_metrics(pred, by, pred_col).sort_values(metric, ascending=False)
    fig, ax = plt.subplots(figsize=(7.5, 0.55 * len(tab) + 1.6))
    ax.barh(range(len(tab)), tab[metric], height=0.6, color=C1, **BAR)
    for i, (v, n, npos) in enumerate(zip(tab[metric], tab["n"], tab["n_pos"])):
        ax.annotate(f"{v:.3f}   n={n:,} ({npos} pos)", xy=(v, i), xytext=(6, 0),
                    textcoords="offset points", va="center", color=INK_2, fontsize=8)
    ax.set_yticks(range(len(tab)), tab[by].astype(str), fontsize=9)
    ax.invert_yaxis()
    _ax(ax, title or f"{SMOKE}{metric} by {by}", metric, "")
    ax.set_xlim(0, max(1.0, float(tab[metric].max()) * 1.45))
    plt.tight_layout(); path = _save(fig, f"slices-{by}"); plt.show()
    return path


def plot_gates(pred, by="language", title=""):
    """Per-instance gate values (§4.4) — thesis-discussion material. A gate that
    barely varies per instance is not doing per-instance gating."""
    cols = [c for c in pred.columns if c.startswith("gate_")]
    if not cols:
        print("no gate columns — baseline or single-channel run")
        return None
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    for c, colour in zip(cols, (C1, C2, C3)):
        axes[0].hist(pred[c], bins=24, color=colour, alpha=0.55,
                     label=c.replace("gate_", ""))
    _ax(axes[0], f"{SMOKE}gate value distribution", "gate weight", "rows")
    _legend_row(axes[0], ncol=len(cols), drop=0.24)
    grouped = pred.groupby(by)[cols].mean()
    x = np.arange(len(grouped)); w = 0.8 / max(len(cols), 1)
    for k, (c, colour) in enumerate(zip(cols, (C1, C2, C3))):
        bars = axes[1].bar(x + k * w - 0.4 + w / 2, grouped[c], width=w * 0.86,
                           color=colour, label=c.replace("gate_", ""), **BAR)
        axes[1].bar_label(bars, fmt="%.2f", fontsize=7, color=INK_2, padding=2)
    axes[1].set_xticks(x, grouped.index.astype(str), fontsize=8)
    _ax(axes[1], f"{SMOKE}mean gate by {by}", "", "gate weight")
    _headroom(axes[1], grouped.to_numpy().ravel())
    _legend_row(axes[1], ncol=len(cols), drop=0.24)
    plt.tight_layout(); path = _save(fig, "gates"); plt.show()
    print(pred[cols].describe().loc[["mean", "std", "min", "max"]].round(4))
    return path


def plot_bootstrap(boot, label="full − baseline", title=""):
    """Paired-bootstrap ΔF1. The question is whether the CI clears zero."""
    diffs = np.asarray(boot["diffs"])
    lo, hi = boot["ci95"]
    fig, ax = plt.subplots(figsize=(7, 3.2))
    ax.hist(diffs, bins=40, color=C1, alpha=0.75)
    ax.axvline(0, color=MUTED, lw=1.5, ls="--")
    ax.axvline(boot["observed"], color=C2, lw=2)
    ax.axvspan(lo, hi, color=C2, alpha=0.12)
    ax.annotate(f"observed {boot['observed']:+.4f}\nCI95 [{lo:+.4f}, {hi:+.4f}]\n"
                f"p = {boot['p']:.3f}", xy=(0.99, 0.95), xycoords="axes fraction",
                ha="right", va="top", color=INK, fontsize=9)
    _ax(ax, title or f"{SMOKE}paired bootstrap ΔF1 ({label})", "ΔF1", "resamples")
    plt.tight_layout(); path = _save(fig, "bootstrap"); plt.show()
    return path


def plot_rq3(report, metric="macro_f1", title=""):
    """RQ3 — pre- vs post-sarcasm sentiment. The slice that matters is
    `sarcastic ∧ literal≠intended`; everything else should barely move."""
    runs = list(dict.fromkeys(report["run"]))
    slices = list(dict.fromkeys(report["slice"]))
    fig, ax = plt.subplots(figsize=(10.5, 4.2))
    x = np.arange(len(slices)); w = 0.8 / max(len(runs), 1)
    tops = []
    for k, (run, colour) in enumerate(zip(runs, (C1, C2, C3))):
        sub = report[report["run"] == run].set_index("slice").reindex(slices)
        vals = sub[metric].fillna(0).to_numpy()
        tops.append(vals)
        bars = ax.bar(x + k * w - 0.4 + w / 2, vals, width=w * 0.86,
                      color=colour, label=run.replace("SMOKE ", ""), **BAR)
        ax.bar_label(bars, fmt="%.2f", fontsize=7, color=INK_2, padding=2)
    # long slice names get wrapped rather than run off the figure
    ax.set_xticks(x, ["\n".join(textwrap.wrap(
        s.replace("language=", "").replace("gold_sarcastic=", "sarc="), 17,
        break_long_words=False)) for s in slices], rotation=0, fontsize=8)
    _ax(ax, title or f"{SMOKE}RQ3 sentiment {metric} by slice", "", metric)
    _headroom(ax, np.concatenate(tops))
    _legend_row(ax, ncol=len(runs), drop=0.34)
    plt.tight_layout(); path = _save(fig, f"rq3-{metric}"); plt.show()
    return path


print(f"plot helpers ready — figures land in {RESULTS / 'figures'}")


## 9 · Smoke test 1 — overfit 16 rows to ~zero loss (§10)

All 8 pilot positives + 8 negatives. A healthy architecture must memorize 16
rows; failure means wiring bugs, not data problems. Run for the baseline
(target-only) and the full model (conv+temp+ret).

In [ ]:
FIG_PREFIX, SMOKE = "SMOKE-", "SMOKE "   # §9–§11 are harness checks whatever the gate says


def overfit_smoke(cfg: Config, n=16, max_steps=150, target_loss=0.05, seed=13):
    cfg = copy.deepcopy(cfg)
    cfg.smoke = True  # overfitting is never a real result
    set_seed(seed)
    # half positives, half negatives — with 1,601 positives, "the first 16
    # rows" would all be sarcastic and a constant predictor would pass
    chosen = (df.loc[sarcastic, "reddit_fullname"].tolist()[:n // 2]
              + df.loc[~sarcastic, "reddit_fullname"].tolist()[:n - n // 2])
    retrieval = (build_fold_retrieval(build_embeddings(cfg), chosen, chosen, k=cfg.retrieval_k)
                 if cfg.use_ret else None)
    loader = DataLoader(SarcasmDataset(chosen, retrieval, build_temporal(cfg)),
                        batch_size=cfg.batch_size,
                        shuffle=True, generator=torch.Generator().manual_seed(seed),
                        collate_fn=Collator(cfg, Tokenize(cfg)))
    model = ContextAwareSarcasmModel(cfg).to(DEVICE)
    opt_cfg = copy.deepcopy(cfg)
    opt_cfg.lr_heads = 1e-3  # hotter head lr for memorization
    optimizer = build_optimizer(model, opt_cfg)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.fp16)
    w = torch.tensor([1.0, 1.0], device=DEVICE)
    losses, step = [], 0
    model.train()
    while step < max_steps:
        for batch in loader:
            if step >= max_steps:
                break
            batch = to_device(batch, DEVICE)
            with torch.autocast("cuda", enabled=cfg.fp16):
                loss = compute_loss(model(batch), batch, cfg, w)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.item()))
            step += 1
            if len(losses) >= 5 and np.mean(losses[-5:]) < target_loss:
                step = max_steps
                break
    final = float(np.mean(losses[-5:]))
    n_pos = sum(LABEL_MAPS["adjudicated"][f] for f in chosen)
    print(f"{cfg.tag()}overfit-{len(chosen)} ({n_pos} pos / {len(chosen) - n_pos} neg): "
          f"final mean loss {final:.4f} ({len(losses)} steps) → {'PASS' if final < target_loss else 'FAIL'}")
    del model
    torch.cuda.empty_cache()
    return {"losses": losses, "passed": final < target_loss}


ov_base = overfit_smoke(smoke_cfg())
ov_full = overfit_smoke(smoke_cfg(use_conv=True, use_temp=True, use_ret=True))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
for ax, ov, title, colour in ((axes[0], ov_base, "baseline (target-only)", C1),
                              (axes[1], ov_full, "full model (conv+temp+ret)", C2)):
    ax.plot(ov["losses"], color=colour, lw=2, solid_capstyle="round")
    ax.axhline(0.05, color=MUTED, lw=1.5, ls="--")
    ax.annotate("target 0.05", xy=(0.99, 0.05), xycoords=("axes fraction", "data"),
                xytext=(0, 4), textcoords="offset points", ha="right",
                color=INK_2, fontsize=8)
    _ax(ax, f"{SMOKE}overfit-16 — {title}", "step", "loss")
plt.tight_layout(); _save(fig, "overfit-16"); plt.show()
assert ov_base["passed"] and ov_full["passed"], "architecture failed to memorize 16 rows"

## 10 · Smoke test 2 — tiny-settings 5-fold dry run (baseline) + pooling check

In [ ]:
for pooling in ("mean", "cls"):  # §4.1 asks to verify both — one fold each at SMOKE scale
    cfg_p = smoke_cfg(pooling=pooling, run_name=f"smoke-pooling-{pooling}")
    set_seed(13)
    tok_p = Tokenize(cfg_p)
    loaders = make_loaders(cfg_p, FOLDS[0], tok_p, 13)
    model = ContextAwareSarcasmModel(cfg_p)
    hist = train_model(model, loaders["train"], loaders["val"], cfg_p, class_weights(FOLDS[0]["train"]))
    pred = predict(model, loaders["test"], cfg_p)
    m = safe_metrics(pred["y_true"].to_numpy(), pred["pred"].to_numpy())
    print(f"{SMOKE}pooling={pooling}: fold-0 test F1={m['f1']:.4f} (n_pos={m['n_pos']})")
    del model
    torch.cuda.empty_cache()
print(f"{SMOKE}both pooling paths run; thesis default stays pooling=mean.\n")

res_base = run_cv(smoke_cfg(run_name="smoke-baseline"))
print(f"\n{SMOKE}5-fold dry run, mean ± std across folds (1 seed, tiny settings):")
print(fold_summary(res_base["fold_metrics"]).to_string(index=False))
print(f"\n{SMOKE}§7.2 hygiene split (pilot has no keyword_oversampled rows — path exercised):")
print(natural_and_all(res_base["predictions"]).to_string(index=False))
print(f"\n{SMOKE}F1 by language (empty-positive folds are the pilot's stratification failure):")
print(slice_metrics(res_base["predictions"], "language").to_string(index=False))
print(f"\n{SMOKE}confusion (rows=true, cols=pred):")
print(confusion_matrix(res_base["predictions"]["y_true"], res_base["predictions"]["pred"], labels=[0, 1]))

# §8b — the four panels that say whether this run is any good
plot_training_curves(res_base["histories"])
plot_fold_spread(res_base["fold_metrics"])
plot_confusion(res_base["predictions"])
BEST_THRESHOLD = plot_threshold_sweep(res_base["predictions"])
plot_pr_roc(res_base["predictions"])
plot_calibration(res_base["predictions"])
plot_slices(res_base["predictions"], "language")

## 11 · Full context model — gates logged, checkpoint round-trip

Condition 8 on fold 0 at SMOKE settings. Per-instance gate values are logged
from day one (§4.4) — their distribution by language / record_type is
thesis-discussion material. Features `[t ; c_fused]` are kept for §13.

In [ ]:
cfg_full = smoke_cfg(use_conv=True, use_temp=True, use_ret=True, run_name="smoke-full")
set_seed(13)
tok_full = Tokenize(cfg_full)
loaders = make_loaders(cfg_full, FOLDS[0], tok_full, 13)
w0 = class_weights(FOLDS[0]["train"])
print(f"{SMOKE}fold-0 class weights (neg, pos): ({w0[0]:.3f}, {w0[1]:.3f})")
model_full = ContextAwareSarcasmModel(cfg_full)
hist = train_model(model_full, loaders["train"], loaders["val"], cfg_full, w0, log_every=1)
print(f"{SMOKE}best val F1: {hist['best_val_f1']:.4f} | temporal λ: "
      f"{float(model_full.temporal_lambda.detach()):.4f} per hour "
      f"(learnable — thesis §3.4.1; init {cfg_full.temporal_lambda_init:.4f})")

pred_f0 = predict(model_full, loaders["test"], cfg_full).merge(META, on="reddit_fullname")
gate_cols = [c for c in pred_f0.columns if c.startswith("gate_")]
print(f"\n{SMOKE}gate distribution over fold-0 test rows:")
print(pred_f0[gate_cols].describe().loc[["mean", "std", "min", "max"]])
print(f"\n{SMOKE}mean gates by language:")
print(pred_f0.groupby("language")[gate_cols].mean())
plot_gates(pred_f0, by="language")
plot_gates(pred_f0, by="record_type")

ckpt = CACHE / "smoke_full_ckpt.pt"
torch.save(model_full.state_dict(), ckpt)
model_check = ContextAwareSarcasmModel(cfg_full)
model_check.load_state_dict(torch.load(ckpt, weights_only=True))
model_check.to(DEVICE)
pred_check = predict(model_check, loaders["test"], cfg_full)
assert np.allclose(pred_f0["prob"].to_numpy(), pred_check["prob"].to_numpy(), atol=1e-6)
print(f"checkpoint round-trip OK → {ckpt}")
del model_check
torch.cuda.empty_cache()

# frozen features + sarcasm flags for ALL rows (fold-0 banks) — used by §13
all_names = df["reddit_fullname"].tolist()
_ret_all = build_fold_retrieval(build_embeddings(cfg_full), FOLDS[0]["train"], all_names,
                                k=cfg_full.retrieval_k, label_source=cfg_full.label_source)
loader_all = DataLoader(SarcasmDataset(all_names, _ret_all, build_temporal(cfg_full)),
                        batch_size=cfg_full.batch_size,
                        shuffle=False, collate_fn=Collator(cfg_full, tok_full))
pred_all = predict(model_full, loader_all, cfg_full, collect_features=True)
FEATURES = pred_all.attrs["features"]
TARGET_EMB = pred_all.attrs["target_emb"]
FLAGS = dict(zip(pred_all["reddit_fullname"], pred_all["pred"].astype(bool)))
PROBS = dict(zip(pred_all["reddit_fullname"], pred_all["prob"]))
print(f"{SMOKE}features {FEATURES.shape} + flags for all rows "
      f"({int(pred_all['pred'].sum())} flagged sarcastic — smoke-quality)")
del model_full
torch.cuda.empty_cache()
FIG_PREFIX, SMOKE = ("SMOKE-", "SMOKE ") if CFG.smoke else ("", "")   # back to what the gate says

## 12 · Ablation matrix — 8 conditions × 5 folds (RQ2, §7.3) + significance

Real settings once the §10 gate passes (smoke settings otherwise, as a harness
check). Staged per [docs/FABILE_BRIEF.md](docs/FABILE_BRIEF.md) §2 — **A**
conditions 1 and 8 (the RQ1 headline), **B** the remaining six, **C** seeds 42
and 7 on the headline pair. Finished runs are re-loaded from `results/`, so
this cell is safe to re-run after any stage; `LEISCHE_STAGES` (set by
`tools/run_notebook.py --stages`) limits what trains now. **E** adds the brief's
§4 improvement rows (annotator-majority label, auxiliary cue heads,
label-quality weighting) — each an added row against an unchanged reference.

Two decisions are reported side by side and neither replaces the other: the
committed 0.5 threshold (`f1@0.5`) and the F1-optimal threshold chosen **per
fold on validation rows** (`f1@val-thr`, §9.8). AUPRC is the threshold-free
view. All of it is agreement with the LLM-ensemble labels.

In [ ]:
CONDITIONS = [("1_baseline", 0, 0, 0), ("2_conv", 1, 0, 0), ("3_temp", 0, 1, 0),
              ("4_ret", 0, 0, 1), ("5_conv_temp", 1, 1, 0), ("6_conv_ret", 1, 0, 1),
              ("7_temp_ret", 0, 1, 1), ("8_full", 1, 1, 1)]
FLAGS_OF = {name: dict(use_conv=bool(c), use_temp=bool(t_), use_ret=bool(r_))
            for name, c, t_, r_ in CONDITIONS}

# Staged plan (docs/FABILE_BRIEF.md §2). Every stage runs on the same frozen
# folds; a finished run is re-loaded from results/ rather than retrained, so
# this cell is safe to re-run after any stage. LEISCHE_STAGES (set by
# tools/run_notebook.py --stages) restricts which stages train NOW; whatever is
# already on disk is always picked up for the table.
STAGE_PLAN = {
    "A": [("1_baseline", 13), ("8_full", 13)],                       # RQ1 headline
    "B": [(n, 13) for n in ("2_conv", "3_temp", "4_ret",              # rest of RQ2
                            "5_conv_temp", "6_conv_ret", "7_temp_ret")],
    "C": [(n, s) for s in (42, 7) for n in ("1_baseline", "8_full")],  # seed variance
}
STAGES_NOW = [s for s in os.environ.get("LEISCHE_STAGES", "A,B,C,D,E").upper().split(",") if s]
REAL = GATE.passed   # otherwise the matrix runs at smoke settings, as a harness check


def condition_cfg(name, seed) -> Config:
    if not REAL:
        return ablation_cfg(run_name=f"ablation-smoke-{name}", **FLAGS_OF[name])
    run_name = f"ablation-{name}" + ("" if seed == 13 else f"-seed{seed}")
    return real_cfg(run_name=run_name, seeds=[seed], save_checkpoint=(seed == 13), **FLAGS_OF[name])


ablation_runs = {}
for stage, items in STAGE_PLAN.items():
    if stage not in STAGES_NOW:
        continue
    print(f"\n=== stage {stage}: {[f'{n}/s{s}' for n, s in items]} "
          f"({'REAL' if REAL else 'SMOKE'} settings) ===", flush=True)
    for name, seed in items:
        t0 = time.time()
        cfg_c = condition_cfg(name, seed)
        ablation_runs[(name, seed)] = run_or_load(cfg_c, verbose=True)
        fm = ablation_runs[(name, seed)]["fold_metrics"]
        print(f"{SMOKE}{name:<12} seed {seed} | F1@0.5 {fm['f1'].mean():.4f} ± {fm['f1'].std():.4f} "
              f"| F1@val-thr {fm['f1_tuned'].mean():.4f} ± {fm['f1_tuned'].std():.4f} "
              f"| AUPRC {fm['auprc'].mean():.4f} | {(time.time() - t0) / 60:6.1f} min", flush=True)
        if not REAL:
            break   # smoke: one seed is the point
# everything finished earlier joins the table too
for stage, items in STAGE_PLAN.items():
    for name, seed in items:
        if (name, seed) not in ablation_runs and run_exists(condition_cfg(name, seed)):
            ablation_runs[(name, seed)] = load_run(condition_cfg(name, seed))

# one entry per condition, all available seeds pooled (folds × seeds, §11.6)
ablation = {}
for name, *_ in CONDITIONS:
    parts = [v for (n, s), v in ablation_runs.items() if n == name]
    if parts:
        ablation[name] = {"fold_metrics": pd.concat([p["fold_metrics"] for p in parts], ignore_index=True),
                          "predictions": pd.concat([p["predictions"] for p in parts], ignore_index=True),
                          "histories": [h for p in parts for h in p["histories"]]}


def _pm(s):
    return f"{s.mean():.4f} ± {0.0 if np.isnan(s.std()) else s.std():.4f}"


table = pd.DataFrame([{
    "condition": name, "runs": len(res["fold_metrics"]),
    "seeds": len(set(res["fold_metrics"]["seed"])),
    "f1@0.5": _pm(res["fold_metrics"]["f1"]), "f1@val-thr": _pm(res["fold_metrics"]["f1_tuned"]),
    "precision@val-thr": _pm(res["fold_metrics"]["precision_tuned"]),
    "recall@val-thr": _pm(res["fold_metrics"]["recall_tuned"]),
    "auprc": _pm(res["fold_metrics"]["auprc"]), "auroc": _pm(res["fold_metrics"]["auroc"]),
    "accuracy@val-thr": _pm(res["fold_metrics"]["accuracy_tuned"]),
    "mean_thr": f"{res['fold_metrics']['threshold'].mean():.2f}"}
    for name, res in ablation.items()])
if len(table):
    print(f"\n{SMOKE}RQ2 ablation matrix — {'agreement with the LLM-ensemble labels, '
          'not with human judgement (ANNOTATION_PROVENANCE.md)' if REAL else 'HARNESS CHECK ONLY'}:")
    print(table.to_string(index=False))
    table.to_csv(RESULTS / f"{'SMOKE-' if not REAL else ''}ablation-matrix.csv", index=False)
    plot_ablation(ablation, "f1_tuned")
    plot_ablation(ablation, "f1")
    plot_ablation(ablation, "recall_tuned")   # recall is where context is expected to pay
    plot_ablation(ablation, "auprc")
    if "8_full" in ablation:
        plot_fold_spread(ablation["8_full"]["fold_metrics"], "f1_tuned",
                         title=f"{SMOKE}condition 8 (full) across folds × seeds")
        plot_training_curves(ablation["8_full"]["histories"], title="condition 8 (full)")
        plot_gates(ablation["8_full"]["predictions"], by="language")
        plot_gates(ablation["8_full"]["predictions"], by="record_type")
    if "1_baseline" in ablation:
        plot_training_curves(ablation["1_baseline"]["histories"], title="condition 1 (baseline)")
else:
    print("no ablation runs available yet — nothing to tabulate")


# ---- stage E · improvement rows (FABILE_BRIEF §4) --------------------------
# Each is an existing config flag turned on for ONE added row; the reference
# condition above is never changed. The annotator-majority rows retrain on the
# pre-adjudication label (2,747 positives) — if the model behaves the same
# under both labels, the finding does not hinge on the adjudicator
# (ANNOTATION_PROVENANCE.md §6.4).
IMPROVEMENT_ROWS = [
    ("majority-1_baseline", "1_baseline", dict(label_source="annotator_majority")),
    ("majority-8_full", "8_full", dict(label_source="annotator_majority")),
    ("aux-cues-8_full", "8_full", dict(aux_cue_heads=True)),
    ("label-weighting-8_full", "8_full", dict(sample_weighting=True, soft_labels=True)),
]


def improvement_cfg(name, cond, over) -> Config:
    return real_cfg(run_name=f"impr-{name}", seeds=[13], **FLAGS_OF[cond], **over)


def _scored(pred, y_col, pred_col="pred_tuned"):
    """Per-fold metrics of `pred_col` against an alternative label column."""
    return pd.DataFrame([safe_metrics(g[y_col].to_numpy(), g[pred_col].to_numpy())
                         for _, g in pred.groupby(["fold", "seed"])])


improvements = {}
if REAL:
    for name, cond, over in IMPROVEMENT_ROWS:
        cfg_i = improvement_cfg(name, cond, over)
        if "E" in STAGES_NOW or run_exists(cfg_i):
            t0 = time.time()
            improvements[name] = (cond, over, run_or_load(cfg_i, verbose=True))
            fm = improvements[name][2]["fold_metrics"]
            print(f"impr {name:<24} F1@val-thr {fm['f1_tuned'].mean():.4f} ± {fm['f1_tuned'].std():.4f} "
                  f"| AUPRC {fm['auprc'].mean():.4f} | {(time.time() - t0) / 60:6.1f} min", flush=True)
if improvements:
    impr_rows = []
    for name, (cond, over, res) in improvements.items():
        fm, ref = res["fold_metrics"], ablation.get(cond)
        ref_fm = ref["fold_metrics"][ref["fold_metrics"]["seed"] == 13] if ref is not None else None
        row = {"row": name, "reference": cond, "flags": ", ".join(f"{k}={v}" for k, v in over.items()),
               "f1@val-thr": _pm(fm["f1_tuned"]), "auprc": _pm(fm["auprc"]),
               "ref f1@val-thr": _pm(ref_fm["f1_tuned"]) if ref_fm is not None else "—",
               "ref auprc": _pm(ref_fm["auprc"]) if ref_fm is not None else "—",
               "ΔF1": (f"{fm['f1_tuned'].mean() - ref_fm['f1_tuned'].mean():+.4f}"
                       if ref_fm is not None else "—")}
        if over.get("label_source") == "annotator_majority" and ref is not None:
            # trained-on × scored-against: the row above is majority/majority; the
            # reference is adjudicated/adjudicated. Cross-score both so the two
            # models are compared on the SAME labels.
            cross_m = _scored(res["predictions"], "sarcastic_adjudicated")
            cross_r = _scored(ref["predictions"][ref["predictions"]["seed"] == 13], "sarcastic_majority")
            row["f1 vs adjudicated label"] = _pm(cross_m["f1"])
            row["ref f1 vs majority label"] = _pm(cross_r["f1"])
            joined = res["predictions"][["reddit_fullname", "fold", "pred_tuned"]].merge(
                ref["predictions"][ref["predictions"]["seed"] == 13][["reddit_fullname", "fold", "pred_tuned"]],
                on=["reddit_fullname", "fold"], suffixes=("_maj", "_adj"))
            row["prediction agreement"] = f"{(joined['pred_tuned_maj'] == joined['pred_tuned_adj']).mean():.3f}"
        impr_rows.append(row)
    impr_table = pd.DataFrame(impr_rows)
    print("\nimprovement rows (stage E) — each vs its unchanged reference, seed 13, val-tuned threshold:")
    print(impr_table.to_string(index=False))
    impr_table.to_csv(RESULTS / "improvement-rows.csv", index=False)
    for name, (cond, over, res) in improvements.items():
        if over.get("label_source") == "annotator_majority" and name.endswith("8_full"):
            plot_gates(res["predictions"], by="language", title="majority-label full model")


### Significance: condition 8 vs condition 1 (bootstrap, randomization, McNemar)

In [ ]:
def _aligned(a, b, pred_col="pred"):
    return a[["reddit_fullname", "seed", "y_true", pred_col]].merge(
        b[["reddit_fullname", "seed", pred_col]], on=["reddit_fullname", "seed"],
        suffixes=("_a", "_b"))


def paired_bootstrap(a, b, n_boot=1000, seed=13, pred_col="pred"):
    m = _aligned(a, b, pred_col)
    rng = np.random.default_rng(seed)
    y, pa, pb = m["y_true"].to_numpy(), m[f"{pred_col}_a"].to_numpy(), m[f"{pred_col}_b"].to_numpy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        diffs = np.array([f1_score(y[i], pb[i], zero_division=0) - f1_score(y[i], pa[i], zero_division=0)
                          for i in (rng.integers(0, len(y), len(y)) for _ in range(n_boot))])
        obs = f1_score(y, pb, zero_division=0) - f1_score(y, pa, zero_division=0)
    return {"observed": float(obs),
            "ci95": (float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))),
            "p": float(min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))),
            "diffs": diffs.tolist()}   # §8b plot_bootstrap draws the distribution


def approx_randomization(a, b, n_iter=1000, seed=13, pred_col="pred"):
    m = _aligned(a, b, pred_col)
    rng = np.random.default_rng(seed)
    y, pa, pb = m["y_true"].to_numpy(), m[f"{pred_col}_a"].to_numpy(), m[f"{pred_col}_b"].to_numpy()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        obs = abs(f1_score(y, pb, zero_division=0) - f1_score(y, pa, zero_division=0))
        count = sum(1 for _ in range(n_iter)
                    if abs(f1_score(y, np.where(s := rng.random(len(y)) < 0.5, pa, pb), zero_division=0)
                           - f1_score(y, np.where(s, pb, pa), zero_division=0)) >= obs - 1e-12)
    return {"observed": float(obs), "p": float((count + 1) / (n_iter + 1))}


def mcnemar_test(a, b, pred_col="pred"):
    from statsmodels.stats.contingency_tables import mcnemar
    m = _aligned(a, b, pred_col)
    ok_a = (m[f"{pred_col}_a"] == m["y_true"]).to_numpy()
    ok_b = (m[f"{pred_col}_b"] == m["y_true"]).to_numpy()
    tab = [[int((ok_a & ok_b).sum()), int((ok_a & ~ok_b).sum())],
           [int((~ok_a & ok_b).sum()), int((~ok_a & ~ok_b).sum())]]
    res = mcnemar(np.array(tab), exact=True)
    return {"table": tab, "p": float(res.pvalue)}


def significance_block(name_a, name_b, pred_col):
    pa, pb = ablation[name_a]["predictions"], ablation[name_b]["predictions"]
    boot = paired_bootstrap(pa, pb, pred_col=pred_col)
    ar = approx_randomization(pa, pb, pred_col=pred_col)
    mc = mcnemar_test(pa, pb, pred_col=pred_col)
    print(f"{SMOKE}[{pred_col}] {name_b} − {name_a}: paired bootstrap ΔF1 {boot['observed']:+.4f} "
          f"CI95 [{boot['ci95'][0]:+.4f}, {boot['ci95'][1]:+.4f}] p={boot['p']:.3f} | "
          f"approx. randomization p={ar['p']:.3f} | "
          f"McNemar discordants {mc['table'][0][1]} vs {mc['table'][1][0]}: p={mc['p']:.3f}")
    return {"pair": f"{name_b}-{name_a}", "decision": pred_col, "delta_f1": boot["observed"],
            "ci95_lo": boot["ci95"][0], "ci95_hi": boot["ci95"][1], "boot_p": boot["p"],
            "randomization_p": ar["p"], "mcnemar_p": mc["p"],
            "discordant_a_only": mc["table"][0][1], "discordant_b_only": mc["table"][1][0],
            "_boot": boot}


if "1_baseline" in ablation and "8_full" in ablation:
    sig_rows = [significance_block("1_baseline", "8_full", "pred_tuned"),
                significance_block("1_baseline", "8_full", "pred")]
    # every single-channel condition against the baseline, val-threshold decision
    for name in ("2_conv", "3_temp", "4_ret", "5_conv_temp", "6_conv_ret", "7_temp_ret"):
        if name in ablation:
            sig_rows.append(significance_block("1_baseline", name, "pred_tuned"))
    sig_table = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in sig_rows])
    sig_table.to_csv(RESULTS / f"{'SMOKE-' if not REAL else ''}significance.csv", index=False)
    plot_bootstrap(sig_rows[0]["_boot"], label="condition 8 − condition 1, val-tuned threshold")
    if not REAL:
        print(f"{SMOKE}at smoke settings these p-values are meaningless — the deliverable is that the machinery runs.")

    pb = ablation["8_full"]["predictions"]
    for by in ("resolved_by", "language", "record_type", "sarcasm_votes"):
        print(f"\n{SMOKE}condition-8 F1 (val-tuned threshold) by {by} (§7.4):")
        print(slice_metrics(pb, by, pred_col="pred_tuned").to_string(index=False))
        plot_slices(pb, by, pred_col="pred_tuned")
    print(f"\n{SMOKE}condition 8 — §7.2 hygiene (no keyword-oversampled rows exist; path exercised):")
    print(natural_and_all(pb, pred_col="pred_tuned").to_string(index=False))
    # threshold / calibration / PR panels for the headline pair
    for name in ("1_baseline", "8_full"):
        p = ablation[name]["predictions"]
        print(f"\n{SMOKE}{name}: threshold sweep on TEST rows is for the picture only — "
              f"the reported thresholds were chosen per fold on VALIDATION rows "
              f"(mean {p['threshold'].mean():.2f})")
        plot_threshold_sweep(p, title=f"{SMOKE}{name} — metric vs threshold (test rows, illustration)")
        plot_pr_roc(p, title=name)
        plot_confusion(p, pred_col="pred_tuned", title=name)
        plot_calibration(p, title=f"{SMOKE}{name} reliability (raw)")
        plot_calibration(p, prob_col="prob_cal", title=f"{SMOKE}{name} reliability (temperature-scaled)")
else:
    print("stage A (conditions 1 and 8) not available yet — significance tests skipped")

### 12b · Specification variants (docs/METHODOLOGY_REVIEW.md) — stage D

Three places where the manuscript and the pipeline disagreed are resolved by
*reporting both*, not by picking silently. Full model (conv+temp+ret), same
frozen folds, real settings. `manuscript-default` **is** condition 8 and is
reused, never retrained.

| variant | what it tests |
|---|---|
| `manuscript-default` | thesis §3.4 as written: k=3, 48 h window, ≤5 posts, learnable λ |
| `retrieval-k5` / `-k10` | §3.4(3) says k=3 is "subject to hyperparameter tuning" |
| `retrieval-xlmr-cls` | §3.4(3) literally — frozen XLM-R `[CLS]` instead of a sentence encoder |
| `temporal-unbounded` | drops the 48 h bound (coverage 36.7% → ~46% of rows) |
| `temporal-lambda-fixed` | λ frozen at init, i.e. what MODEL_PLAN §4.2 wrongly called the thesis baseline |

In [ ]:
SPEC_VARIANTS = [
    ("manuscript-default", {}),
    ("retrieval-k5", dict(retrieval_k=5)),
    ("retrieval-k10", dict(retrieval_k=10)),
    ("retrieval-xlmr-cls", dict(retrieval_encoder="xlmr_cls")),
    ("temporal-unbounded", dict(temporal_window_hours=None, temporal_k=10)),
    ("temporal-lambda-fixed", dict(temporal_lambda_learnable=False)),
]


def variant_cfg(name, over) -> Config:
    if not REAL:
        return ablation_cfg(use_conv=True, use_temp=True, use_ret=True, run_name=f"spec-{name}", **over)
    if name == "manuscript-default":      # identical to condition 8 — reuse, never retrain
        return condition_cfg("8_full", 13)
    return real_cfg(use_conv=True, use_temp=True, use_ret=True, run_name=f"spec-{name}", **over)


spec_rows, spec_runs = [], {}
for name, over in SPEC_VARIANTS:
    cfg_v = variant_cfg(name, over)
    if "D" in STAGES_NOW or run_exists(cfg_v):
        t0 = time.time()
        res = run_or_load(cfg_v, verbose=True)
        spec_runs[name] = res
        fm = res["fold_metrics"]
        cov = np.mean([len(v) > 0 for v in build_temporal(cfg_v).values()])
        spec_rows.append({
            "variant": name, "runs": len(fm),
            "f1@0.5": _pm(fm["f1"]), "f1@val-thr": _pm(fm["f1_tuned"]),
            "recall@val-thr": _pm(fm["recall_tuned"]), "auprc": _pm(fm["auprc"]),
            "temporal_coverage": f"{cov:.1%}",
            "retrieval_k": cfg_v.retrieval_k, "encoder": cfg_v.retrieval_encoder,
            "window_h": cfg_v.temporal_window_hours, "lambda_learnable": cfg_v.temporal_lambda_learnable,
            "min": round((time.time() - t0) / 60, 1)})
        print(f"{SMOKE}{name:<22} F1@val-thr {fm['f1_tuned'].mean():.4f} | AUPRC {fm['auprc'].mean():.4f} "
              f"| temporal coverage {cov:.1%} | {(time.time() - t0) / 60:6.1f} min", flush=True)
    else:
        print(f"{name:<22} skipped (stage D not requested and no finished run on disk)")

if spec_rows:
    spec_table = pd.DataFrame(spec_rows)
    print(f"\n{SMOKE}specification-variant comparison"
          f"{' — HARNESS CHECK ONLY' if not REAL else ' (agreement with the LLM-ensemble labels)'}:")
    print(spec_table.to_string(index=False))
    spec_table.to_csv(RESULTS / f"{'SMOKE-' if not REAL else ''}spec-variants.csv", index=False)
    print("\nthe temporal_coverage column is the one to carry into the manuscript: "
          "the 48 h bound is a data-availability constraint, not a modelling result.")

## 13 · RQ3 — two-stage sentiment evaluation (§8, thesis §3.5)

Ground truth `labels.intended_sentiment`; rows whose sentiment vote never
resolved are excluded (they carry no ground truth).

- **Stage 1 (pre-sarcasm)** — thesis §3.5: a sentiment head trained on the
  annotated `literal_sentiment` and applied *directly to the target text*,
  i.e. over the **target-only** embedding, no context. (MODEL_PLAN §8 proposed
  the shipped `aux.tx_sentiment` instead; uyam does not collect it, so the
  manuscript's own formulation is what runs — handoff H3.)
- **Stage 2 (post-sarcasm)** — rows the sarcasm model flags are re-predicted by
  a head over the frozen `[t ; c_fused]` features trained on
  `intended_sentiment`. Unflagged rows keep their stage-1 prediction.

Both heads are fit on fold-0 **training** rows only, so RQ3 supervision never
touches the sarcasm encoder or the test fold. The interesting cell:
*sarcastic ∧ literal≠intended*.

In [ ]:
literal = df["labels"].map(lambda l: l["literal_sentiment"])
intended = df["labels"].map(lambda l: l["intended_sentiment"])
has_sent = literal.notna() & intended.notna()
shift = has_sent & (literal != intended)
print(f"rows with a resolved sentiment pair: {int(has_sent.sum())} / {len(df)}")
print(f"literal≠intended: {int(shift.sum())} | sarcastic ∧ shift: "
      f"{int((sarcastic & shift).sum())} rows")

# ---- which sarcasm model feeds stage 2 ------------------------------------
# The real fold-0 checkpoint that §12 stage A saved (condition 8, seed 13),
# with the decision threshold chosen on ITS validation fold. Falls back to the
# §11 smoke model when no real checkpoint exists, so the cell always runs.
all_names = df["reddit_fullname"].tolist()
_ck = CHECKPOINTS / "ablation-8_full-fold0-seed13.pt"
if GATE.passed and _ck.exists():
    _payload = torch.load(_ck, weights_only=False, map_location="cpu")
    assert _payload["dataset_identity"] == IDENTITY, "checkpoint was trained on a different export"
    cfg_rq3 = Config(**_payload["config"])
    model_rq3 = ContextAwareSarcasmModel(cfg_rq3)
    model_rq3.load_state_dict(_payload["state_dict"])
    model_rq3.to(DEVICE)
    _ret_rq3 = build_fold_retrieval(build_embeddings(cfg_rq3), FOLDS[0]["train"], all_names,
                                    k=cfg_rq3.retrieval_k, label_source=cfg_rq3.label_source)
    loader_rq3 = DataLoader(SarcasmDataset(all_names, _ret_rq3, build_temporal(cfg_rq3), label_map(cfg_rq3)),
                            batch_size=64, shuffle=False, collate_fn=Collator(cfg_rq3, Tokenize(cfg_rq3)))
    pred_all = predict(model_rq3, loader_rq3, cfg_rq3, collect_features=True)
    FEATURES, TARGET_EMB = pred_all.attrs["features"], pred_all.attrs["target_emb"]
    RQ3_THR, RQ3_TEMP = float(_payload["threshold"]), float(_payload["temperature"])
    PROBS = dict(zip(pred_all["reddit_fullname"],
                     1.0 / (1.0 + np.exp(-pred_all["margin"].to_numpy() / RQ3_TEMP))))
    FLAGS = dict(zip(pred_all["reddit_fullname"], pred_all["prob"] >= RQ3_THR))
    RQ3_TAG = ""
    print(f"\nRQ3 sarcasm model: {_ck.name} (condition 8, fold 0, seed {_payload['seed']}) | "
          f"val-tuned threshold {RQ3_THR:.2f}, temperature {RQ3_TEMP:.2f} | "
          f"{int(sum(FLAGS.values()))} of {len(FLAGS)} rows flagged")
    del model_rq3
    torch.cuda.empty_cache()
else:
    RQ3_TAG = SMOKE
    print(f"\n{SMOKE}RQ3 sarcasm model: §11 smoke model (no real fold-0 checkpoint yet)")


def sentiment_metrics(y_true, y_pred):
    if len(y_true) == 0:  # small slices can be empty — NaN, not a crash
        return {"macro_f1": float("nan"), "accuracy": float("nan"), "n": 0}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return {"macro_f1": float(f1_score(y_true, y_pred, labels=list(SENTIMENTS),
                                           average="macro", zero_division=0)),
                "accuracy": float(accuracy_score(y_true, y_pred)), "n": int(len(y_true))}


def rq3_report(sub: pd.DataFrame, pred: pd.Series, tag: str) -> pd.DataFrame:
    y = sub["labels"].map(lambda l: l["intended_sentiment"])
    lg = sub["labels"].map(lambda l: l["language"])
    sc = sub["labels"].map(lambda l: l["sarcastic"])
    lit = sub["labels"].map(lambda l: l["literal_sentiment"])
    rows = [{"slice": "overall", **sentiment_metrics(y, pred)}]
    rows += [{"slice": f"language={v}", **sentiment_metrics(y[lg == v], pred[lg == v])}
             for v in sorted(lg.unique())]
    rows += [{"slice": f"gold_sarcastic={s}", **sentiment_metrics(y[sc == s], pred[sc == s])}
             for s in (False, True)]
    m = sc & (lit != y)
    rows.append({"slice": "sarcastic ∧ literal≠intended", **sentiment_metrics(y[m], pred[m])})
    out = pd.DataFrame(rows)
    out.insert(0, "run", tag)
    return out


# RQ3 population: fold-0 rows that actually carry sentiment ground truth
feat_of = {f: i for i, f in enumerate(pred_all["reddit_fullname"])}
_has_sent_of = dict(zip(df["reddit_fullname"], has_sent))
_literal_of = dict(zip(df["reddit_fullname"], literal))
_intended_of = dict(zip(df["reddit_fullname"], intended))
rq3_train = [f for f in FOLDS[0]["train"] if _has_sent_of[f]]
test_df = df[df["reddit_fullname"].isin(FOLDS[0]["test"]) & has_sent]
print(f"\n{RQ3_TAG}RQ3 fold-0: {len(rq3_train)} train rows, {len(test_df)} test rows "
      f"(dropped rows without a resolved sentiment pair)")

# stage 1 — thesis §3.5: trained on literal_sentiment, applied to the TARGET
# TEXT ALONE, so it is fit over the context-free target embedding
stage1_head = LogisticRegression(max_iter=2000)
stage1_head.fit(TARGET_EMB[[feat_of[f] for f in rq3_train]], [_literal_of[f] for f in rq3_train])
s1 = pd.Series(stage1_head.predict(TARGET_EMB[[feat_of[f] for f in test_df["reddit_fullname"]]]),
               index=test_df.index)

# external cross-check (MODEL_PLAN §8 candidate a): the shipped tx_sentiment
# probabilities are model-independent, so they bound how much of any stage-1→2
# gain is our own head rather than sarcasm-awareness. Reported, not primary.
S1_EXTERNAL = None
if HAS_TX:
    S1_EXTERNAL = test_df["aux"].map(lambda a: max(
        {"positive": a["tx_sentiment"]["p_pos"], "neutral": a["tx_sentiment"]["p_neu"],
         "negative": a["tx_sentiment"]["p_neg"]}.items(), key=lambda kv: kv[1])[0])

# stage 2 — trained on intended_sentiment over the frozen [t ; c_fused] features
stage2_head = LogisticRegression(max_iter=2000)
stage2_head.fit(FEATURES[[feat_of[f] for f in rq3_train]], [_intended_of[f] for f in rq3_train])
s2 = pd.Series(stage2_head.predict(FEATURES[[feat_of[f] for f in test_df["reddit_fullname"]]]),
               index=test_df.index)
if not RQ3_TAG:   # the demo re-uses these two heads next to the real checkpoint (cache/ is gitignored)
    import pickle
    with open(CHECKPOINTS / "rq3-heads-fold0.pkl", "wb") as _fh:
        pickle.dump({"stage1_literal_on_target_emb": stage1_head, "stage2_intended_on_features": stage2_head,
                     "checkpoint": _ck.name, "threshold": RQ3_THR, "temperature": RQ3_TEMP,
                     "dataset_identity": IDENTITY}, _fh)

flagged = pd.Series([bool(FLAGS[f]) for f in test_df["reddit_fullname"]], index=test_df.index)
final_pred = s2.where(flagged, s1)
print(f"{RQ3_TAG}{int(flagged.sum())} of {len(test_df)} test rows flagged sarcastic "
      "→ re-interpreted by the stage-2 head")
# oracle bound: stage 2 fired on exactly the gold-sarcastic rows — how much of
# the gap is the flag, and how much is the stage-2 head itself
oracle = pd.Series(test_df["labels"].map(lambda l: bool(l["sarcastic"])).to_numpy(), index=test_df.index)
oracle_pred = s2.where(oracle, s1)

print(f"\n{RQ3_TAG}RQ3 comparison on fold-0 test rows (ground truth = intended_sentiment):")
_reports = [rq3_report(test_df, s1, RQ3_TAG + "stage1-only"),
            rq3_report(test_df, final_pred, RQ3_TAG + "two-stage"),
            rq3_report(test_df, oracle_pred, RQ3_TAG + "two-stage (oracle flag)"),
            rq3_report(test_df, s2, RQ3_TAG + "stage2-everywhere")]
if S1_EXTERNAL is not None:
    _reports.insert(0, rq3_report(test_df, S1_EXTERNAL, RQ3_TAG + "stage1-external(tx)"))
    # MODEL_PLAN §8 candidate (a) as the stage-1 reader, our stage-2 head on the
    # flagged rows: isolates what the sarcasm flag adds on top of the best
    # available context-free sentiment reading
    _reports.append(rq3_report(test_df, s2.where(flagged, S1_EXTERNAL),
                               RQ3_TAG + "two-stage (external stage 1)"))
_rq3 = pd.concat(_reports, ignore_index=True)
print(_rq3.to_string(index=False))
_rq3.to_csv(RESULTS / f"{'SMOKE-' if RQ3_TAG else ''}rq3-fold0.csv", index=False)
plot_rq3(_rq3[_rq3["run"].isin([RQ3_TAG + "stage1-only", RQ3_TAG + "two-stage",
                                RQ3_TAG + "two-stage (oracle flag)"])], "macro_f1")
plot_rq3(_rq3[_rq3["run"].isin([RQ3_TAG + "stage1-only", RQ3_TAG + "two-stage",
                                RQ3_TAG + "two-stage (oracle flag)"])], "accuracy")

y_sarc = test_df["labels"].map(lambda l: int(l["sarcastic"])).to_numpy()
probs_f0 = np.array([PROBS[f] for f in test_df["reddit_fullname"]])
print(f"\n{RQ3_TAG}sarcasm-flag ECE on fold-0 test (temperature-scaled prob): {ece(y_sarc, probs_f0):.4f} | "
      f"flag precision {safe_metrics(y_sarc, flagged.to_numpy().astype(int))['precision']:.3f}, "
      f"recall {safe_metrics(y_sarc, flagged.to_numpy().astype(int))['recall']:.3f}")

## 14 · Verdict & next steps

What ran, what it means and what to do next is logged stage by stage in
[docs/RESULTS_LOG.md](docs/RESULTS_LOG.md). The gate below is the standing
caveat: training is legitimate, the labels are not yet validated against human
judgement, so every number is agreement with the LLM ensemble.

In [ ]:
print(f"{SMOKE}smoke test 1 (overfit-16, baseline + full): PASS")
print(f"{SMOKE}smoke test 2 (5-fold dry run): PASS")
print(f"{SMOKE}8×5 ablation matrix + significance tests: RAN END-TO-END")
print(f"{SMOKE}RQ3 two-stage pipeline: RAN END-TO-END")
print()
print(GATE.render())